# Bitcoin Transaction Fraud Detection: Feature Engineering and Modelling

## Summary: Results & Main Findings

| File | Model | Public ROC-AUC |
|---|---|---|
| (project baseline) | NA | about 0.92 |
| `sub1_baseline.csv` | LightGBM on the raw features | 0.94894 |
| `sub2_stack2.csv` | LightGBM + neighbour-score stacking ("stack2") | 0.95157 |
| `sub3_blend.csv` | Rank average of sub1 and sub2 | 0.95094 |
| `sub4_stack2_shallow.csv` | Stack2 with a shallow final model (experiment, rejected) | 0.94725 |
| **`sub5_smote_x4.csv`** | LightGBM on the raw features + per-time-step SMOTE x4 | **0.95591** |
| `sub6_blend_stack2_smote.csv` | Rank average of stack2 and sub5 | 0.95469 |

**Final entries: sub5 and sub6.** Only these two files are written by the notebook (sub1 to sub3 are commented out in section 12). All files average 3 seeds (`QUICK = True`, see "How to run").

1. Almost all of the gain over the random-forest baseline (about 0.92 to 0.949) comes from switching to LightGBM. Neighbour-score stacking adds +0.0026 on the public board, and per-time-step SMOTE on the plain model adds +0.0070 (0.95591). The two do not add up.
2. The graph is strongly homophilic, hence neighbour predictions are informative. The test graph is much sparser than the training graph, hence structural features (PageRank, k-core, component size) drift and hurt. Masking the neighbour features during training handles the missing neighbours.
3. Not helpful: class weighting, rank normalisation, recency weights, dropping drifting features, tuned LightGBM settings, self-training, unlabelled rows as negatives, sliding windows, richer neighbour or component features, a second stacking round, GNNs (GCN / SAGE) and blends of tree models. Undersampling, duplicating illicit rows and SMOTE across time steps also failed; only SMOTE within a single time step helped (section 10.8).

## Contents
1. Setup and data
2. Validation design
3. Evaluation harness
4. Baselines
5. Drift handling
6. Graph feature engineering
7. Graph diagnostics
8. Neighbour-score stacking (the final method)
9. Ensembles of tree models
10. Further experiments: LightGBM settings, self-training, richer neighbour features, unlabelled data and sparse graphs, GNNs (rejected); time-aware resampling (per-step SMOTE adopted)
11. Frozen holdout, threshold tuning, error analysis
12. Final fit and submission files
13. Decision log

## How to run
* Set `DATA_DIR` in the setup cell if the data folder is not found automatically. Run the notebook top to bottom ("Restart & Run All").
* `QUICK = True` is the setting used for the submitted files: one seed in the single-run comparisons (three seeds for the headline comparisons), `N_TREES = 150` (LightGBM and XGBoost use 2 x `N_TREES` = 300 trees) and three seeds in the final fit. `QUICK = False` uses 400 trees and five final seeds and is slower; numbers will shift slightly.

## 1. Setup and data

In [1]:
# !pip install xgboost lightgbm optuna
# !pip install torch torch_geometric
# !pip install gensim

In [1]:
import os, time, warnings, math
import numpy as np, pandas as pd
import scipy.sparse as sp
import matplotlib.pyplot as plt, seaborn as sns
import networkx as nx
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.inspection import permutation_importance
from sklearn.metrics import (roc_auc_score, average_precision_score,
                             precision_recall_fscore_support, precision_recall_curve)

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

try:
    import xgboost as xgb; HAS_XGB = True
except ImportError:
    HAS_XGB = False
try:
    import lightgbm as lgb; HAS_LGBM = True
except ImportError:
    HAS_LGBM = False
try:
    import optuna; HAS_OPTUNA = True
except ImportError:
    HAS_OPTUNA = False
try:
    import torch, torch.nn as nn
    from torch_geometric.nn import GATConv, GCNConv, SAGEConv, DeepGraphInfomax
    HAS_PYG = True
except Exception:
    HAS_PYG = False
print(f"xgboost={HAS_XGB} lightgbm={HAS_LGBM} optuna={HAS_OPTUNA} torch_geometric={HAS_PYG}")

# Data directory detection
DATA_DIR = None
for _c in ["../data", "./data", "data", "../../data", "."]:
    if os.path.exists(os.path.join(_c, "train.csv")):
        DATA_DIR = _c
        break
if DATA_DIR is None:
    raise FileNotFoundError(f"train.csv not found from {os.getcwd()}. Set DATA_DIR to the folder that contains it.")

# Submission CSVs and the experiment logs are written here
OUT_DIR = os.path.dirname(os.path.abspath(DATA_DIR)) if os.path.abspath(DATA_DIR) != os.getcwd() else "."
print("DATA_DIR =", os.path.abspath(DATA_DIR), "| outputs ->", os.path.abspath(OUT_DIR))

QUICK = True #True = faster run (setting used for submitted files)
SEEDS = (0,) if QUICK else (0, 1, 2) # random seeds used in each experiment; scores are averaged over them
RUN_BETWEENNESS = False           # skipped (not used)
GNN_EPOCHS = 80 if QUICK else 200 # default training length for the GNN code; section 10.6 sets its own value (120)
N_TREES = 150 if QUICK else 400 # base number of trees; LightGBM/XGBoost use 2 x N_TREES (= 300 when QUICK)
RANDOM_STATE = 42 # fixed seed for the one-off random sample used in the drift check

xgboost=True lightgbm=True optuna=True torch_geometric=True
DATA_DIR = c:\Users\erina\Downloads\bt4012_kaggle competition\data | outputs -> c:\Users\erina\Downloads\bt4012_kaggle competition


In [2]:
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test = pd.read_csv(f"{DATA_DIR}/test.csv")
edges = pd.read_csv(f"{DATA_DIR}/txs_edgelist.csv")
sample_sub = pd.read_csv(f"{DATA_DIR}/sample_submission.csv")

feat_all = [c for c in train.columns if c.startswith("feat_")]
test = test.copy()
if "label" not in test.columns:
    test["label"] = np.nan

# One table for train + test; so that graph features use both
ALL = pd.concat([train.assign(is_test=0), test.assign(is_test=1, label=np.nan)], ignore_index=True)
N = len(ALL) # total number of transactions
T = ALL["time_step"].values # time steps for each transaction
Y = ALL["label"].values                        # NaN = unknown (and all test rows)
IS_TEST = ALL["is_test"].values == 1 # true for test rows
LAB = np.where(~np.isnan(Y) & ~IS_TEST)[0]     # labelled training rows
UNK = np.where(np.isnan(Y) & ~IS_TEST)[0]      # unlabelled training-period rows
TEST_POS = np.where(IS_TEST)[0] # test rows

print(f"rows={N:,}  labelled={len(LAB):,}  unknown(train)={len(UNK):,}  test={len(TEST_POS):,}")
print(f"train steps {T[~IS_TEST].min()}-{T[~IS_TEST].max()}, test steps {T[IS_TEST].min()}-{T[IS_TEST].max()}")
print("illicit rate among labelled:", round(float(np.mean(Y[LAB])), 4))
print("NaNs in features:", int(ALL[feat_all].isna().sum().sum()))

rows=157,101  labelled=31,235  unknown(train)=110,537  test=15,329
train steps 1-35, test steps 36-49
illicit rate among labelled: 0.1167
NaNs in features: 0


In [3]:
# Remove exact-duplicate feature columns (correlation ~ 1), computed on a sample.
samp = ALL[feat_all].sample(20000, random_state=RANDOM_STATE)
corr = samp.corr().abs()
upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
dup_cols = [c for c in upper.columns if (upper[c] >= 0.99999).any()]
feat = [c for c in feat_all if c not in dup_cols]
print(f"dropping {len(dup_cols)} duplicate columns -> {len(feat)} features")

# Feature matrices
X_raw = ALL[feat].values.astype(np.float32)
# Per-time-step percentile rank: each feature is replaced by its rank (0-1) within its own time step; attempt to reduce drift, not used in final model
X_rank = ALL.groupby("time_step")[feat].rank(pct=True).values.astype(np.float32)
print(X_raw.shape, X_rank.shape)

dropping 8 duplicate columns -> 157 features
(157101, 157) (157101, 157)


## 2. Validation design

Each fold is `(train_end, val_lo, val_hi)`: train on labelled rows with `time_step <= train_end`, validate on labelled rows in `[val_lo, val_hi]`.

* **roll**: expanding window, validation right after training.
* **gap**: validation sits several steps after the training window.
* **HOLDOUT**: frozen (validation set).

In [4]:
DEV_FOLDS = {
    "roll": [(14, 15, 20), (20, 21, 26), (26, 27, 30)],
    "gap":  [(12, 19, 24), (18, 25, 30)],
}
HOLDOUT = (25, 31, 35)

def fold_pos(f): # turn a fold definition into row positions
    te, lo, hi = f
    tr = LAB[T[LAB] <= te]
    va = LAB[(T[LAB] >= lo) & (T[LAB] <= hi)]
    return tr, va

rows = []
for s, fl in list(DEV_FOLDS.items()) + [("holdout", [HOLDOUT])]:
    for k, f in enumerate(fl):
        tr, va = fold_pos(f)
        rows.append(dict(set=s, fold=k, train_steps=f"1-{f[0]}", val_steps=f"{f[1]}-{f[2]}",
                         n_train=len(tr), n_val=len(va),
                         train_illicit=Y[tr].mean().round(3), val_illicit=Y[va].mean().round(3),
                         val_pos=int(Y[va].sum())))
pd.DataFrame(rows)

,set,fold,train_steps,val_steps,n_train,n_val,train_illicit,val_illicit,val_pos
0,roll,0,1-14,15-20,14896,3993,0.067,0.192,766
1,roll,1,1-20,21-26,18889,5828,0.094,0.114,662
2,roll,2,1-26,27-30,24717,2188,0.098,0.238,521
3,gap,0,1-12,19-24,13670,6362,0.049,0.124,788
4,gap,1,1-18,25-30,17244,3299,0.083,0.223,735
5,holdout,0,1-25,31-35,24200,4330,0.097,0.159,690


## 3. Evaluation harness

Every model is a function `fp(tr_pos, va_pos, seed) -> scores on va_pos`. The same code path is used for validation and for the submission.

*Illicit precision / recall / F1 are shown at the default 0.5 threshold, and `f1_best` is the best F1 over all thresholds on that fold (optimistic diagnostic, not a selection metric).*

In [5]:
LEADER = []          # one summary row per experiment (model name, dev/gap scores, noise, runtime)
PRED = {}            # PRED[name][(set, fold)]; mean score over seeds
BASELINE = None      # name of the reference model

def illicit_prf(y, s, thr=0.5): # Precision, recall and F1 for the illicit class, using threshold thr on the scores.
    pred = (s >= thr).astype(int)
    p, r, f, _ = precision_recall_fscore_support(y, pred, average="binary", zero_division=0)
    return p, r, f

def best_f1(y, s): # Best F1 over every possible threshold
    p, r, _ = precision_recall_curve(y, s)
    return float(np.max(2 * p * r / (p + r + 1e-12)))

def evaluate(name, fp, seeds=SEEDS, sets=("roll", "gap"), verbose=True): # Score the model fp on every fold of the chosen sets, for every seed, and log a summary row
    t0 = time.time()
    rows, PRED[name] = [], {}
    for s in sets:
        for k, f in enumerate(DEV_FOLDS[s]): # each fold in the set
            tr, va = fold_pos(f) # training rows and validation rows for this fold
            y = Y[va].astype(int) # true labels of the validation rows
            all_scores = []
            for seed in seeds: # repeat with different seeds
                sc = np.asarray(fp(tr, va, seed), dtype=np.float64)
                all_scores.append(sc)
                p, r, f1 = illicit_prf(y, sc)
                rows.append(dict(set=s, fold=k, seed=seed,
                                 roc_auc=roc_auc_score(y, sc), pr_auc=average_precision_score(y, sc),
                                 prec=p, rec=r, f1=f1, f1_best=best_f1(y, sc)))
            PRED[name][(s, k)] = np.mean(all_scores, axis=0) # average over seeds
    df = pd.DataFrame(rows)
    g = df.groupby(["set", "fold"])[["roc_auc", "pr_auc", "prec", "rec", "f1", "f1_best"]].mean()
    out = dict(model=name,
               roc_dev=g["roc_auc"].mean(),
               roc_roll=g.loc["roll"]["roc_auc"].mean() if "roll" in sets else np.nan,
               roc_gap=g.loc["gap"]["roc_auc"].mean() if "gap" in sets else np.nan,
               fold_std=g["roc_auc"].std(),
               seed_std=df.groupby(["set", "fold"])["roc_auc"].std().mean() if len(seeds) > 1 else np.nan,
               pr_dev=g["pr_auc"].mean(), prec=g["prec"].mean(), rec=g["rec"].mean(),
               f1=g["f1"].mean(), f1_best=g["f1_best"].mean(), secs=round(time.time() - t0))
    LEADER.append(out)
    if verbose:
        d = ""
        if BASELINE is not None: # show the change versus the reference model
            b = next((r for r in LEADER if r["model"] == BASELINE), None)
            if b is not None:
                d = f" | d_gap={out['roc_gap'] - b['roc_gap']:+.4f} d_dev={out['roc_dev'] - b['roc_dev']:+.4f}"
        print(f"{name:48s} dev={out['roc_dev']:.4f} gap={out['roc_gap']:.4f} roll={out['roc_roll']:.4f} "
              f"fold_std={out['fold_std']:.4f} pr={out['pr_dev']:.4f} ({out['secs']}s){d}")
    return out

def leaderboard(top=None): # Table of all experiments, best gap score first (ties broken by dev score). Re-run models keep only their latest row
    df = pd.DataFrame(LEADER).drop_duplicates("model", keep="last")
    df = df.sort_values(["roc_gap", "roc_dev"], ascending=False).reset_index(drop=True)
    return df.head(top) if top else df

# Holdout evaluation
HOLD = []
def holdout_eval(name, fp, seeds=SEEDS):
    tr, va = fold_pos(HOLDOUT)
    y = Y[va].astype(int)
    scs = [np.asarray(fp(tr, va, sd), dtype=np.float64) for sd in seeds]
    PRED.setdefault(name, {})["holdout"] = np.mean(scs, axis=0)
    rocs = [roc_auc_score(y, s) for s in scs]
    sc = PRED[name]["holdout"]
    p, r, f1 = illicit_prf(y, sc)
    row = dict(model=name, roc_auc=np.mean(rocs), pr_auc=average_precision_score(y, sc),
               prec=p, rec=r, f1=f1, f1_best=best_f1(y, sc))
    HOLD.append(row)
    print(f"[HOLDOUT] {name:44s} roc={row['roc_auc']:.4f} pr={row['pr_auc']:.4f} f1_best={row['f1_best']:.4f}")
    return row

## 4. Baselines

Logistic regression, random forest, XGBoost and LightGBM on the raw features, each with and without class weighting.

In [6]:
# Build a fresh model from `factory` and train it on (Xtr, ytr)
def fit_model(factory, Xtr, ytr, w, seed, weighted):
    # scale_pos_weight = (# licit) / (# illicit). Only used when `weighted=True`; ignored by default
    # class weighting made every model worse on the gap folds
    spw = (ytr == 0).sum() / max((ytr == 1).sum(), 1) # max(..., 1) avoids division by zero
    m = factory(seed, spw, weighted)
    if w is None:
        m.fit(Xtr, ytr)
    elif isinstance(m, Pipeline):
        m.fit(Xtr, ytr, **{m.steps[-1][0] + "__sample_weight": w})
    else:
        m.fit(Xtr, ytr, sample_weight=w)
    return m

# Model factories: each returns a fresh, untrained model
# All take (seed, spw, weighted)
def make_lr(seed, spw, weighted):
    return make_pipeline(StandardScaler(),
        LogisticRegression(max_iter=2000, C=0.1, class_weight="balanced" if weighted else None, random_state=seed))

def make_rf(seed, spw, weighted):
    return RandomForestClassifier(n_estimators=N_TREES, n_jobs=-1, random_state=seed,
                                  class_weight="balanced_subsample" if weighted else None)

def make_hgb(seed, spw, weighted):
    return HistGradientBoostingClassifier(random_state=seed, class_weight="balanced" if weighted else None)

def make_xgb(seed, spw, weighted):
    return xgb.XGBClassifier(n_estimators=N_TREES * 2, learning_rate=0.05, max_depth=6, subsample=0.8,
                             colsample_bytree=0.8, tree_method="hist", eval_metric="logloss",
                             scale_pos_weight=spw if weighted else 1.0, random_state=seed, n_jobs=-1)

def make_lgbm(seed, spw, weighted):
    return lgb.LGBMClassifier(n_estimators=N_TREES * 2, learning_rate=0.05, num_leaves=31, subsample=0.8,
                              subsample_freq=1, colsample_bytree=0.8,
                              scale_pos_weight=spw if weighted else 1.0, random_state=seed, n_jobs=-1, verbose=-1)

# Wrap a model factory + feature matrix into the harness interface fp(tr, va, seed) -> scores for va rows
# # decay<1 down-weights older time steps; w = decay**(latest_train_step - step); decay=None means equal weights
def gbdt_fp(factory, X, weighted=False, decay=None):
    def fp(tr, va, seed):
        y = Y[tr].astype(int)
        w = None if decay is None else decay ** (T[tr].max() - T[tr])
        m = fit_model(factory, X[tr], y, w, seed, weighted)
        return m.predict_proba(X[va])[:, 1]
    return fp

# Registry of available models
FACTORIES = {"LR": make_lr, "RF": make_rf, "HGB": make_hgb}
if HAS_XGB:  FACTORIES["XGB"] = make_xgb
if HAS_LGBM: FACTORIES["LGBM"] = make_lgbm
MAIN = "LGBM" if HAS_LGBM else ("XGB" if HAS_XGB else "HGB")
MAIN_FACTORY = FACTORIES[MAIN]
print("main tree model:", MAIN)

main tree model: LGBM


In [8]:
# Baselines: score every model on the raw features, then score again with class weights
# `d_gap` / `d_dev` are the change versus the baseline `MAIN | raw`. Compare with `fold_std`. If a change is smaller than the fold std, it is noise and should not be kept just because it is positive.
for n, f in FACTORIES.items():
    evaluate(f"{n} | raw", gbdt_fp(f, X_raw))
BASELINE = f"{MAIN} | raw"
for n in [k for k in ["LR", "RF", "XGB", "LGBM"] if k in FACTORIES]:
    evaluate(f"{n} | raw | class-weighted", gbdt_fp(FACTORIES[n], X_raw, weighted=True))
leaderboard()

LR | raw                                         dev=0.8840 gap=0.8469 roll=0.9088 fold_std=0.0403 pr=0.5832 (1s)
RF | raw                                         dev=0.9124 gap=0.8539 roll=0.9513 fold_std=0.0851 pr=0.8847 (5s)
HGB | raw                                        dev=0.9361 gap=0.8744 roll=0.9772 fold_std=0.0630 pr=0.8947 (7s)
XGB | raw                                        dev=0.9466 gap=0.9027 roll=0.9758 fold_std=0.0499 pr=0.9028 (18s)
LGBM | raw                                       dev=0.9490 gap=0.9048 roll=0.9784 fold_std=0.0546 pr=0.9048 (8s)
LR | raw | class-weighted                        dev=0.8909 gap=0.8540 roll=0.9155 fold_std=0.0449 pr=0.5791 (2s) | d_gap=-0.0508 d_dev=-0.0581
RF | raw | class-weighted                        dev=0.9152 gap=0.8506 roll=0.9582 fold_std=0.0820 pr=0.8828 (6s) | d_gap=-0.0542 d_dev=-0.0338
XGB | raw | class-weighted                       dev=0.9367 gap=0.8817 roll=0.9734 fold_std=0.0606 pr=0.8990 (18s) | d_gap=-0.0231 d_dev=-0.0

,model,roc_dev,roc_roll,roc_gap,fold_std,seed_std,pr_dev,prec,rec,f1,f1_best,secs
0,LGBM | raw,0.948957,0.978375,0.904829,0.054624,NaN,0.904786,0.977424,0.759984,0.850067,0.879476,8
1,XGB | raw,0.946587,0.975835,0.902716,0.049942,NaN,0.902831,0.969946,0.778081,0.858948,0.876083,18
2,XGB | raw | class-weighted,0.936728,0.973396,0.881725,0.060565,NaN,0.898982,0.944020,0.804422,0.864489,0.872899,18
3,LGBM | raw | class-weighted,0.937913,0.979723,0.875197,0.071588,NaN,0.899964,0.965487,0.788940,0.863577,0.878970,9
4,HGB | raw,0.936081,0.977230,0.874357,0.063003,NaN,0.894708,0.966366,0.768609,0.851597,0.868931,7
5,LR | raw | class-weighted,0.890863,0.915451,0.853982,0.044864,NaN,0.579068,0.389128,0.890175,0.536487,0.640688,2
6,RF | raw,0.912371,0.951336,0.853924,0.085141,NaN,0.884652,0.987084,0.760932,0.854495,0.872858,5
7,RF | raw | class-weighted,0.915160,0.958205,0.850591,0.082020,NaN,0.882817,0.987508,0.704646,0.817011,0.864749,6
8,LR | raw,0.884025,0.908768,0.846909,0.040272,NaN,0.583202,0.535453,0.676045,0.587796,0.613082,1


### Findings
* LightGBM (dev 0.9490, gap 0.9048) and XGBoost (0.9466, 0.9027) are clearly ahead of HistGradientBoosting (0.9361, 0.8744), random forest (0.9124, 0.8539) and logistic regression (0.8840, 0.8469). **LightGBM without class weights is the main model.**
* Class weighting made the two gradient-boosted models worse on the gap folds, each against its own unweighted run (LightGBM −0.030, XGBoost −0.021). It made no clear difference for random forest (−0.003) and slightly helped logistic regression (+0.007), both within fold noise. Weights change the scores' scale but not their ranking quality, which is all ROC-AUC measures, and here they mostly added noise. LightGBM without class weights stays the main model.

## 5. Drift handling

Tested three fixes to handle drift
1. **Adversarial validation**: how separable are train rows from test rows, and which features give it away?
2. **Per-step rank normalisation** of the features.
3. **Recency weighting** of training rows.

In [9]:
# Adversarial validation: can a model tell train rows from test rows?
# Idea: label train rows 0 and test rows 1, then try to predict that label from the features.
# AUC ~0.5 = the two sets look the same. AUC near 1.0 = they look very different (strong drift).

def adversarial_auc(X, n=40000, seed=0):
    rng = np.random.RandomState(seed)
    idx = rng.choice(N, size=min(n, N), replace=False)
    y_adv = IS_TEST[idx].astype(int)
    clf = RandomForestClassifier(n_estimators=100, max_depth=8, n_jobs=-1, random_state=seed)
    p = cross_val_predict(clf, X[idx], y_adv, cv=StratifiedKFold(3, shuffle=True, random_state=seed),
                          method="predict_proba")[:, 1]
    return roc_auc_score(y_adv, p)

print("adversarial AUC, raw features :", round(adversarial_auc(X_raw), 4))
print("adversarial AUC, rank features:", round(adversarial_auc(X_rank), 4), "(lower = less drift)")

# Which features drift the most?
# Fit the same kind of model on all rows and read off its feature importances: the features it relies on most to separate train from test are the ones that drift most.
adv_rf = RandomForestClassifier(n_estimators=100, max_depth=8, n_jobs=-1, random_state=0).fit(X_raw, IS_TEST.astype(int))
drift_rank = pd.Series(adv_rf.feature_importances_, index=feat).sort_values(ascending=False)
drift_rank.head(10)

adversarial AUC, raw features : 1.0
adversarial AUC, rank features: 1.0 (lower = less drift)


feat_100    0.186873
feat_101    0.169996
feat_103    0.169086
feat_136    0.168853
feat_139    0.073883
feat_137    0.036560
feat_2      0.024424
feat_109    0.017881
feat_107    0.016502
feat_113    0.015330
dtype: float64

In [10]:
# Four ways to deal with drift (all compared with LightGBM on raw features)

# 1) Rank-normalise: replace each feature by its rank within its own time step (X_rank)
evaluate(f"{MAIN} | rank-normalised", gbdt_fp(MAIN_FACTORY, X_rank))

# 2) Recency weights: weight a training row by decay ** (steps before the latest training step),
#    so recent time steps count more than old ones. Smaller decay = forgets the past faster.
#    Each decay is tried on both the raw and the rank features.
for d in (0.95, 0.9, 0.8):
    evaluate(f"{MAIN} | raw | decay{d}", gbdt_fp(MAIN_FACTORY, X_raw, decay=d))
    evaluate(f"{MAIN} | rank | decay{d}", gbdt_fp(MAIN_FACTORY, X_rank, decay=d))

# 3) Drop the k most drifting features (ranked by the adversarial model in the previous cell).
for k in (6, 20):
    keep = [i for i, c in enumerate(feat) if c not in set(drift_rank.index[:k])]
    evaluate(f"{MAIN} | raw | drop top{k} drift feats", gbdt_fp(MAIN_FACTORY, X_raw[:, keep]))

# Top 10 of the leaderboard, to see how each variant compares with the reference "LGBM | raw"
leaderboard(10)

LGBM | rank-normalised                           dev=0.9361 gap=0.8995 roll=0.9605 fold_std=0.0470 pr=0.8646 (8s) | d_gap=-0.0053 d_dev=-0.0129
LGBM | raw | decay0.95                           dev=0.9508 gap=0.9103 roll=0.9778 fold_std=0.0488 pr=0.9055 (9s) | d_gap=+0.0055 d_dev=+0.0019
LGBM | rank | decay0.95                          dev=0.9352 gap=0.9005 roll=0.9584 fold_std=0.0476 pr=0.8611 (8s) | d_gap=-0.0043 d_dev=-0.0137
LGBM | raw | decay0.9                            dev=0.9496 gap=0.9104 roll=0.9758 fold_std=0.0479 pr=0.9059 (9s) | d_gap=+0.0056 d_dev=+0.0007
LGBM | rank | decay0.9                           dev=0.9368 gap=0.8994 roll=0.9617 fold_std=0.0460 pr=0.8587 (8s) | d_gap=-0.0055 d_dev=-0.0122
LGBM | raw | decay0.8                            dev=0.9428 gap=0.8993 roll=0.9718 fold_std=0.0599 pr=0.9024 (9s) | d_gap=-0.0056 d_dev=-0.0062
LGBM | rank | decay0.8                           dev=0.9369 gap=0.9025 roll=0.9599 fold_std=0.0478 pr=0.8679 (8s) | d_gap=-0.0023 d_dev=

,model,roc_dev,roc_roll,roc_gap,fold_std,seed_std,pr_dev,prec,rec,f1,f1_best,secs
0,LGBM | raw | decay0.9,0.949634,0.975765,0.910439,0.047889,NaN,0.905858,0.978196,0.764077,0.853367,0.879845,9
1,LGBM | raw | decay0.95,0.950830,0.977837,0.910319,0.048825,NaN,0.905475,0.978652,0.762078,0.851528,0.879394,9
2,LGBM | raw,0.948957,0.978375,0.904829,0.054624,NaN,0.904786,0.977424,0.759984,0.850067,0.879476,8
3,LGBM | raw | drop top20 drift feats,0.948778,0.978865,0.903648,0.050295,NaN,0.904969,0.982422,0.768435,0.856871,0.875953,8
4,XGB | raw,0.946587,0.975835,0.902716,0.049942,NaN,0.902831,0.969946,0.778081,0.858948,0.876083,18
5,LGBM | rank | decay0.8,0.936938,0.959901,0.902493,0.047790,NaN,0.867868,0.980775,0.509784,0.659124,0.826058,8
6,LGBM | raw | drop top6 drift feats,0.946189,0.975602,0.902070,0.053338,NaN,0.904834,0.984842,0.761311,0.853844,0.883425,9
7,LGBM | rank | decay0.95,0.935232,0.958366,0.900529,0.047603,NaN,0.861062,0.979883,0.479755,0.624819,0.815927,8
8,LGBM | rank-normalised,0.936102,0.960500,0.899506,0.047008,NaN,0.864615,0.981511,0.474588,0.621339,0.817704,8
9,LGBM | rank | decay0.9,0.936770,0.961717,0.899350,0.046031,NaN,0.858727,0.977452,0.486787,0.631334,0.813592,8


In [11]:
# Fold-by-fold ROC-AUC of model a vs model b (both must already have been run through evaluate())
def paired(a, b):
    rows = []
    for s in ("roll", "gap"):
        for k, f in enumerate(DEV_FOLDS[s]):
            _, va = fold_pos(f); y = Y[va].astype(int)
            rows.append(dict(set=s, fold=k, a=roc_auc_score(y, PRED[a][(s, k)]), b=roc_auc_score(y, PRED[b][(s, k)])))
    d = pd.DataFrame(rows); d["diff"] = d.a - d.b
    print(d.round(4)); print(f"{a!r} wins {(d['diff'] > 0).sum()} of {len(d)} folds")

# Is recency weighting (decay 0.95) really better than the plain model?
paired("LGBM | raw | decay0.95", "LGBM | raw")

    set  fold       a       b    diff
0  roll     0  0.9940  0.9940 -0.0000
1  roll     1  0.9417  0.9438 -0.0021
2  roll     2  0.9978  0.9973  0.0005
3   gap     0  0.9427  0.9475 -0.0048
4   gap     1  0.8779  0.8621  0.0158
'LGBM | raw | decay0.95' wins 2 of 5 folds


### Findings
* Adversarial validation separates train from test perfectly (AUC 1.0 on raw and rank-normalised features). The most drifting features are `feat_100`, `feat_101`, `feat_103`, `feat_136` and `feat_139`.
* Per-step rank normalisation: dev -0.013, gap -0.005. Dropping the top 6 / top 20 drifting features: gap -0.003 / -0.001. Recency weights of 0.9 and 0.95 gave about +0.0055 on gap, but the fold-by-fold check for 0.95 shows it wins only 2 of 5 folds (gap fold 1 improves by +0.0158 while gap fold 0 gets worse by 0.0048), which is noise.
* So the main model keeps the **raw features with no recency weights** (the cell below).

In [7]:
X_BASE = X_raw          # or X_rank
DECAY = None            # or 0.95 / 0.9 / 0.8
BASE_TAG = "raw"        # label used in experiment names below

## 6. Graph feature engineering

In [8]:
# Map transaction IDs to row positions
id2pos = pd.Series(np.arange(N), index=ALL["txId"].values)
es = edges["txId1"].map(id2pos); et = edges["txId2"].map(id2pos)
ok = es.notna() & et.notna()
src = es[ok].astype(int).values; dst = et[ok].astype(int).values
print(f"edges: {len(edges):,} total, {ok.sum():,} between known transactions")

# Adjacency matrices (N x N, mostly zeros, stored sparsely)
A_dir = sp.csr_matrix((np.ones(len(src), dtype=np.float32), (src, dst)), shape=(N, N))
A_dir.sum_duplicates(); A_dir.data[:] = 1; A_dir.setdiag(0); A_dir.eliminate_zeros()
A_in = A_dir.T.tocsr()                                         # row i = predecessors of i
A_und = ((A_dir + A_dir.T) > 0).astype(np.float32).tocsr()

def row_max(M, v, fill=-1.0):
    M = M.tocsr(); out = np.full(M.shape[0], fill, dtype=np.float64)
    nz = np.diff(M.indptr) > 0
    if nz.any():
        out[nz] = np.maximum.reduceat(v[M.indices], M.indptr[:-1][nz])
    return out

def pagerank(A, alpha=0.85, iters=60):
    n = A.shape[0]; out = np.asarray(A.sum(1)).ravel() # out-degree of every node
    inv = np.divide(1.0, out, out=np.zeros_like(out, dtype=np.float64), where=out > 0) # 1/out-degree, 0 for dead ends
    P = sp.diags(inv) @ A # row-normalised transition matrix
    r = np.full(n, 1.0 / n) # start with equal importance
    for _ in range(iters):
        # new score = alpha * (importance received along edges + spread from dead-end nodes) + (1-alpha) random jump
        r = alpha * (P.T @ r + r[out == 0].sum() / n) + (1 - alpha) / n
    return r * n # rescale so the average score is 1

edges: 234,355 total, 175,918 between known transactions


In [9]:
GF = {}   # graph feature name -> array of length N

# Basic degree counts
in_deg = np.asarray(A_dir.sum(0)).ravel(); out_deg = np.asarray(A_dir.sum(1)).ravel()
und_deg = np.asarray(A_und.sum(1)).ravel()
# Same features using ALL edges in the edge list, including edges to transactions that are in neither file
in_full = edges["txId2"].value_counts().reindex(ALL["txId"].values).fillna(0).values
out_full = edges["txId1"].value_counts().reindex(ALL["txId"].values).fillna(0).values

# group: degree
GF.update(in_deg=in_deg, out_deg=out_deg, tot_deg=in_deg + out_deg,
          out_in_ratio=(out_deg + 1) / (in_deg + 1),
          in_deg_full=in_full, out_deg_full=out_full,
          phantom_nbrs=(in_full + out_full) - (in_deg + out_deg))

# group: neighbourhood structure
GF["nbr_deg_mean"] = np.where(und_deg > 0, (A_und @ und_deg) / np.maximum(und_deg, 1), 0)
GF["nbr_deg_max"] = row_max(A_und, und_deg, fill=0)
A2 = (A_und @ A_und).tocsr(); A2.setdiag(0); A2.eliminate_zeros()
GF["two_hop_count"] = np.diff(A2.indptr).astype(float)
GF["succ_out_deg_sum"] = A_dir @ out_deg          # downstream fan-out
GF["pred_in_deg_sum"] = A_in @ in_deg             # upstream fan-in
del A2

# group: pagerank (forward and reverse)
GF["pagerank"] = pagerank(A_dir)
GF["pagerank_rev"] = pagerank(A_in)

# group: k-core (Higher = deeper inside a densely connected cluster)
G_und = nx.from_scipy_sparse_array(A_und)
core = nx.core_number(G_und)
GF["kcore"] = np.array([core[i] for i in range(N)], dtype=float)

GROUPS = {
    "degree": ["in_deg", "out_deg", "tot_deg", "out_in_ratio", "in_deg_full", "out_deg_full", "phantom_nbrs"],
    "nbr_structure": ["nbr_deg_mean", "nbr_deg_max", "two_hop_count", "succ_out_deg_sum", "pred_in_deg_sum"],
    "pagerank": ["pagerank", "pagerank_rev"],
    "kcore": ["kcore"],
}
if RUN_BETWEENNESS:
    bc = nx.betweenness_centrality(G_und, k=300, seed=0)
    GF["betweenness_approx"] = np.array([bc[i] for i in range(N)])
    GROUPS["betweenness"] = ["betweenness_approx"]

def gf_matrix(names):
    return np.column_stack([GF[n] for n in names]).astype(np.float32)

print({g: len(v) for g, v in GROUPS.items()})

{'degree': 7, 'nbr_structure': 5, 'pagerank': 2, 'kcore': 1}


In [15]:
# Reference model: LightGBM on the base features (raw, no recency weights; BASE_TAG = "raw", DECAY = None).
# Setting BASELINE makes every printout below show d_gap / d_dev, the change versus this model.
BASELINE = f"{MAIN} | {BASE_TAG}" + (f" | decay{DECAY}" if DECAY else "")
evaluate(BASELINE, gbdt_fp(MAIN_FACTORY, X_BASE, decay=DECAY))

# Add ONE group of graph features at a time (degree, nbr_structure, pagerank, kcore) to the base features.
for g, names in GROUPS.items():
    evaluate(f"{MAIN} | {BASE_TAG} + graph:{g}", gbdt_fp(MAIN_FACTORY, np.hstack([X_BASE, gf_matrix(names)]), decay=DECAY))

# Add ALL graph features together. X_GRAPH is kept in case a later cell wants it.
all_g = [n for v in GROUPS.values() for n in v]
X_GRAPH = np.hstack([X_BASE, gf_matrix(all_g)])
evaluate(f"{MAIN} | {BASE_TAG} + graph:ALL", gbdt_fp(MAIN_FACTORY, X_GRAPH, decay=DECAY))

leaderboard(10)

LGBM | raw                                       dev=0.9490 gap=0.9048 roll=0.9784 fold_std=0.0546 pr=0.9048 (9s) | d_gap=+0.0000 d_dev=+0.0000
LGBM | raw + graph:degree                        dev=0.9484 gap=0.9065 roll=0.9763 fold_std=0.0532 pr=0.9043 (9s) | d_gap=+0.0017 d_dev=-0.0006
LGBM | raw + graph:nbr_structure                 dev=0.9478 gap=0.9087 roll=0.9739 fold_std=0.0510 pr=0.9040 (9s) | d_gap=+0.0038 d_dev=-0.0012
LGBM | raw + graph:pagerank                      dev=0.9427 gap=0.8891 roll=0.9784 fold_std=0.0607 pr=0.9025 (9s) | d_gap=-0.0157 d_dev=-0.0063
LGBM | raw + graph:kcore                         dev=0.9408 gap=0.8867 roll=0.9768 fold_std=0.0657 pr=0.9015 (9s) | d_gap=-0.0181 d_dev=-0.0082
LGBM | raw + graph:ALL                           dev=0.9417 gap=0.8904 roll=0.9759 fold_std=0.0642 pr=0.9009 (10s) | d_gap=-0.0144 d_dev=-0.0072


,model,roc_dev,roc_roll,roc_gap,fold_std,seed_std,pr_dev,prec,rec,f1,f1_best,secs
0,LGBM | raw | decay0.9,0.949634,0.975765,0.910439,0.047889,NaN,0.905858,0.978196,0.764077,0.853367,0.879845,9
1,LGBM | raw | decay0.95,0.950830,0.977837,0.910319,0.048825,NaN,0.905475,0.978652,0.762078,0.851528,0.879394,9
2,LGBM | raw + graph:nbr_structure,0.947782,0.973860,0.908664,0.050991,NaN,0.904000,0.973786,0.760347,0.849286,0.878892,9
3,LGBM | raw + graph:degree,0.948369,0.976255,0.906540,0.053165,NaN,0.904251,0.974169,0.764558,0.851778,0.878980,9
4,LGBM | raw,0.948957,0.978375,0.904829,0.054624,NaN,0.904786,0.977424,0.759984,0.850067,0.879476,9
5,LGBM | raw | drop top20 drift feats,0.948778,0.978865,0.903648,0.050295,NaN,0.904969,0.982422,0.768435,0.856871,0.875953,8
6,XGB | raw,0.946587,0.975835,0.902716,0.049942,NaN,0.902831,0.969946,0.778081,0.858948,0.876083,18
7,LGBM | rank | decay0.8,0.936938,0.959901,0.902493,0.047790,NaN,0.867868,0.980775,0.509784,0.659124,0.826058,8
8,LGBM | raw | drop top6 drift feats,0.946189,0.975602,0.902070,0.053338,NaN,0.904834,0.984842,0.761311,0.853844,0.883425,9
9,LGBM | rank | decay0.95,0.935232,0.958366,0.900529,0.047603,NaN,0.861062,0.979883,0.479755,0.624819,0.815927,8


### Findings
* Degree features and neighbourhood structure are within the noise. PageRank, k-core and all groups together hurt the gap folds. The test graph likely has a different shape.
* None of these is used in the final model. The 165 given features already contain one-hop neighbour aggregates.

## 7. Graph diagnostics

TO check the structure of the data

1. **Label coverage:** how many validation nodes can be reached from a labelled training node?
2. **No known neighbours:** how many rows have no neighbour at all?
3. **Homophily:** do illicit transactions link to other illicit ones more than chance (`r` vs `2xy` test)?
4. **Sparsity:** how many neighbours does a row have in the labelled, unknown and test groups?

### 7.1 Label coverage (why label propagation is dropped)

In [16]:
def label_coverage(f, hops=3):
    tr, va = fold_pos(f)
    seed_vec = np.zeros(N); seed_vec[tr] = 1
    reach = seed_vec.copy()
    for _ in range(hops):
        reach = np.minimum(1, reach + A_und @ reach)
    return float(reach[va].mean())

for s, fl in DEV_FOLDS.items():
    for k, f in enumerate(fl):
        print(s, k, "share of validation nodes within 3 hops of a labelled training node:", round(label_coverage(f), 4))

roll 0 share of validation nodes within 3 hops of a labelled training node: 0.0
roll 1 share of validation nodes within 3 hops of a labelled training node: 0.0
roll 2 share of validation nodes within 3 hops of a labelled training node: 0.0
gap 0 share of validation nodes within 3 hops of a labelled training node: 0.0
gap 1 share of validation nodes within 3 hops of a labelled training node: 0.0


### 7.2 Rows with no known neighbours
Every training row has at least one neighbour, but **25.4% of test rows have none**. Neighbour-based features must therefore be allowed to be missing, and the model has to learn to cope. This will be handled by masking.

In [17]:
iso = und_deg == 0
print("no known neighbours | labelled train %.3f | unknown train %.3f | test %.3f"
      % (iso[LAB].mean(), iso[UNK].mean(), iso[TEST_POS].mean()))

no known neighbours | labelled train 0.000 | unknown train 0.000 | test 0.254


### 7.3 Homophily
Among the 23,900 edges joining two labelled transactions, the share that join an illicit and a licit transaction is **r = 0.053**, against **2xy = 0.206** expected if labels were assigned at random. The network is strongly homophilic.

In [18]:
lab_mask = np.zeros(N, bool); lab_mask[LAB] = True
c = A_und.tocoo()
keep = (c.row < c.col) & lab_mask[c.row] & lab_mask[c.col]
ya, yb = Y[c.row[keep]].astype(int), Y[c.col[keep]].astype(int)
y_frac = Y[LAB].astype(int).mean()
print("labelled-labelled edges:", keep.sum())
print("observed cross-label share r =", round((ya != yb).mean(), 3),
      "| expected if random, 2xy =", round(2 * y_frac * (1 - y_frac), 3))

labelled-labelled edges: 23900
observed cross-label share r = 0.053 | expected if random, 2xy = 0.206


### 7.4 Sparsity of the test graph
Labelled rows have 2.87 neighbours on average, unknown rows 2.14 and test rows **1.66**. Labelled transactions are better connected than the rest, so both training and validation rows are denser than the test. The edge-drop rate that would mimic the test density for labelled rows is `Q = 0.421`, used in section 10.5.

In [19]:
print("mean neighbours  labelled: %.2f | unknown: %.2f | test: %.2f" %
      (und_deg[LAB].mean(), und_deg[UNK].mean(), und_deg[TEST_POS].mean()))
Q = float(np.clip(1 - und_deg[TEST_POS].mean() / und_deg[LAB].mean(), 0, 0.9))
print("edge drop rate Q (labelled vs test):", round(Q, 3))

mean neighbours  labelled: 2.87 | unknown: 2.14 | test: 1.66
edge drop rate Q (labelled vs test): 0.421


## 8. Neighbour-score stacking (the final method)

### 8.1 Idea
Stage 1 produces a fraud score for every node. Stage 2 adds that score and its neighbourhood aggregates (mean over predecessors / successors / all neighbours, max, and a 2-hop mean) as features to the final LightGBM.

**Leakage control:** scores for the training nodes are *out-of-fold* (the training window is cut into blocks of time steps; each block is scored by a model fitted on the other blocks), and scores for all other nodes are the average of those models. This is rebuilt from scratch inside every validation fold and inside the final fit.

**Missing neighbours (stack2):** rows with no known neighbours get missing values for the neighbour features. During training 25% of rows have those features blanked at random, and in validation 25% of rows are blanked (`SIM_VAL_MASK`) to mimic the 25.4% of test rows with no neighbours. `SIM_VAL_MASK` is 0.25 for validation and the holdout, and 0.0 for the final fit.

In [10]:
def oof_scores(predict_fn, tr, k=4):
    """predict_fn(fit_pos) -> (N, d) scores for ALL nodes from a model fitted on labelled rows fit_pos.
    Returns stage-1 scores that are safe to use as features:
      - training rows get OUT-OF-FOLD scores (from models that never saw their label)
      - all other rows (validation, test, unlabelled) get the average of the k models' scores."""
    steps = np.unique(T[tr]); blocks = np.array_split(steps, min(k, len(steps))) # cut the training window into k blocks of time steps
    in_tr = np.zeros(N, bool); in_tr[tr] = True
    out = acc = None
    for b in blocks:
        hold = tr[np.isin(T[tr], b)]; fit = tr[~np.isin(T[tr], b)]
        P = np.asarray(predict_fn(fit), dtype=np.float64).reshape(N, -1)
        if out is None:
            out = np.zeros_like(P); acc = np.zeros_like(P)
        out[hold] = P[hold]
        acc[~in_tr] += P[~in_tr]
    out[~in_tr] = acc[~in_tr] / len(blocks)
    return out

# For each node, the mean of `vec` over its neighbours in the graph M. Nodes with no neighbours get -1.
def mean_over(M, vec):
    d = np.asarray(M.sum(1)).ravel()
    return np.where(d > 0, (M @ vec) / np.maximum(d, 1), -1.0)

# Neighbourhood summary of a score p (one value per node): 5 columns.
def nbr_aggregate(p):
    p = np.asarray(p, dtype=np.float64)
    m_in, m_out, m_und = mean_over(A_in, p), mean_over(A_dir, p), mean_over(A_und, p)
    mx = row_max(A_und, p)
    m2 = mean_over(A_und, np.where(m_und >= 0, m_und, 0.0))
    return np.column_stack([m_in, m_out, m_und, mx, m2])

# Stage-1 model: a plain tree model on features X. Returns f(fit, seed) -> (N, 1) fraud scores for all nodes.
def tree_stage1(factory, X):
    def f(fit, seed):
        y = Y[fit].astype(int)
        m = fit_model(factory, X[fit], y, None, seed, False)
        return m.predict_proba(X)[:, 1:2]
    return f

def stacked_fp(stage1, X_base, final_factory, decay=None, k=4, use_nbr=True, weighted=False):
    """Two-stage model in the usual fp(tr, va, seed) form:
    stage 1 -> out-of-fold fraud score for every node; (optionally) neighbour aggregates of that score;
    stage 2 -> final model on [original features, own score, neighbour aggregates]"""
    def fp(tr, va, seed):
        S = oof_scores(lambda fit: stage1(fit, seed), tr, k)
        parts = [X_base, S] + ([nbr_aggregate(S[:, 0])] if use_nbr else [])
        X = np.hstack(parts).astype(np.float32)
        return gbdt_fp(final_factory, X, weighted, decay)(tr, va, seed)
    return fp

# Stack with neighbour aggregates
evaluate(f"{MAIN} | stack: HGB stage1 + nbr preds",
         stacked_fp(tree_stage1(make_hgb, X_BASE), X_BASE, MAIN_FACTORY, decay=DECAY))
# Stack without neighbour aggregates
evaluate(f"{MAIN} | stack: HGB stage1 only (no nbr agg)",
         stacked_fp(tree_stage1(make_hgb, X_BASE), X_BASE, MAIN_FACTORY, decay=DECAY, use_nbr=False))

LGBM | stack: HGB stage1 + nbr preds             dev=0.9538 gap=0.9173 roll=0.9781 fold_std=0.0435 pr=0.9094 (84s)
LGBM | stack: HGB stage1 only (no nbr agg)       dev=0.9453 gap=0.8961 roll=0.9781 fold_std=0.0591 pr=0.9021 (97s)


{'model': 'LGBM | stack: HGB stage1 only (no nbr agg)',
 'roc_dev': np.float64(0.9452843636110094),
 'roc_roll': np.float64(0.9780672285784199),
 'roc_gap': np.float64(0.8961100661598941),
 'fold_std': np.float64(0.059083919440738084),
 'seed_std': nan,
 'pr_dev': np.float64(0.9021292468832762),
 'prec': np.float64(0.9602767190781055),
 'rec': np.float64(0.7695345732693284),
 'f1': np.float64(0.8487793839446258),
 'f1_best': np.float64(0.8757444135480819),
 'secs': 97}

In [11]:
# Share of VALIDATION rows whose neighbour features are blanked out, to imitate the test set (25.4% of test rows have no neighbours).
# 0.25 for validation and the holdout. It must be set to 0.0 before the final fit in section 12.
SIM_VAL_MASK = 0.25

def stacked_fp2(stage1, X_base, final_factory, decay=None, k=4, p_mask=0.25):
    def fp(tr, va, seed):
        S = oof_scores(lambda fit: stage1(fit, seed), tr, k)
        agg = nbr_aggregate(S[:, 0])
        agg[und_deg == 0] = np.nan                 # real isolates: missing, not "-1"
        rng = np.random.RandomState(seed)
        m = np.zeros(N, bool)
        m[tr] = rng.rand(len(tr)) < p_mask         # teach the model to cope with missing neighbours
        m[va] = rng.rand(len(va)) < SIM_VAL_MASK   # mimic the test set's missing neighbours
        agg[m] = np.nan
        X = np.hstack([X_base, S, agg]).astype(np.float32)
        return gbdt_fp(final_factory, X, False, decay)(tr, va, seed)
    return fp

# Ablation of the training mask: does teaching the model about missing neighbours help?
evaluate("LGBM | stack2: no train mask, 25% val masked",
         stacked_fp2(tree_stage1(make_hgb, X_BASE), X_BASE, MAIN_FACTORY, decay=DECAY, p_mask=0.0))
evaluate("LGBM | stack2: train mask .25, 25% val masked",
         stacked_fp2(tree_stage1(make_hgb, X_BASE), X_BASE, MAIN_FACTORY, decay=DECAY, p_mask=0.25))

LGBM | stack2: no train mask, 25% val masked     dev=0.9520 gap=0.9141 roll=0.9774 fold_std=0.0440 pr=0.9066 (104s)
LGBM | stack2: train mask .25, 25% val masked    dev=0.9516 gap=0.9138 roll=0.9768 fold_std=0.0444 pr=0.9077 (96s)


{'model': 'LGBM | stack2: train mask .25, 25% val masked',
 'roc_dev': np.float64(0.9515752839400919),
 'roc_roll': np.float64(0.97678222320574),
 'roc_gap': np.float64(0.91376487504162),
 'fold_std': np.float64(0.044374668460698756),
 'seed_std': nan,
 'pr_dev': np.float64(0.9077440007809507),
 'prec': np.float64(0.9623712682024024),
 'rec': np.float64(0.7799891645905018),
 'f1': np.float64(0.8563976547465568),
 'f1_best': np.float64(0.8809655413003801),
 'secs': 96}

In [22]:
paired("LGBM | stack2: train mask .25, 25% val masked", "LGBM | raw")
evaluate("LGBM | raw [3 seeds]", gbdt_fp(MAIN_FACTORY, X_BASE, decay=DECAY), seeds=(0,1,2))

    set  fold       a       b    diff
0  roll     0  0.9950  0.9940  0.0010
1  roll     1  0.9382  0.9438 -0.0056
2  roll     2  0.9971  0.9973 -0.0002
3   gap     0  0.9347  0.9475 -0.0128
4   gap     1  0.8928  0.8621  0.0306
'LGBM | stack2: train mask .25, 25% val masked' wins 2 of 5 folds
LGBM | raw [3 seeds]                             dev=0.9448 gap=0.8979 roll=0.9761 fold_std=0.0569 pr=0.9028 (37s) | d_gap=-0.0069 d_dev=-0.0041


{'model': 'LGBM | raw [3 seeds]',
 'roc_dev': np.float64(0.9448429264272246),
 'roc_roll': np.float64(0.9761473715832386),
 'roc_gap': np.float64(0.8978862586932035),
 'fold_std': np.float64(0.05689326372221636),
 'seed_std': np.float64(0.004877651183144018),
 'pr_dev': np.float64(0.9027926873919366),
 'prec': np.float64(0.9762455099740783),
 'rec': np.float64(0.766490309572813),
 'f1': np.float64(0.853952707715562),
 'f1_best': np.float64(0.8789223848868805),
 'secs': 37}

In [23]:
evaluate("LGBM | stack2 [3 seeds]",
         stacked_fp2(tree_stage1(make_hgb, X_BASE), X_BASE, MAIN_FACTORY, decay=DECAY, p_mask=0.25), seeds=(0, 1, 2))
evaluate("XGB | raw [3 seeds]", gbdt_fp(make_xgb, X_BASE, decay=DECAY), seeds=(0, 1, 2))
leaderboard(8)

LGBM | stack2 [3 seeds]                          dev=0.9478 gap=0.9059 roll=0.9757 fold_std=0.0508 pr=0.9047 (532s) | d_gap=+0.0011 d_dev=-0.0012
XGB | raw [3 seeds]                              dev=0.9447 gap=0.8987 roll=0.9753 fold_std=0.0521 pr=0.9022 (53s) | d_gap=-0.0061 d_dev=-0.0043


,model,roc_dev,roc_roll,roc_gap,fold_std,seed_std,pr_dev,prec,rec,f1,f1_best,secs
0,LGBM | stack: HGB stage1 + nbr preds,0.953790,0.978117,0.917301,0.043487,NaN,0.909398,0.959326,0.792121,0.862043,0.884128,32
1,"LGBM | stack2: no train mask, 25% val masked",0.952038,0.977356,0.914061,0.044023,NaN,0.906626,0.967312,0.771657,0.853329,0.877135,34
2,"LGBM | stack2: train mask .25, 25% val masked",0.951575,0.976782,0.913765,0.044375,NaN,0.907744,0.962371,0.779989,0.856398,0.880966,35
3,LGBM | raw | decay0.9,0.949634,0.975765,0.910439,0.047889,NaN,0.905858,0.978196,0.764077,0.853367,0.879845,9
4,LGBM | raw | decay0.95,0.950830,0.977837,0.910319,0.048825,NaN,0.905475,0.978652,0.762078,0.851528,0.879394,9
5,LGBM | raw + graph:nbr_structure,0.947782,0.973860,0.908664,0.050991,NaN,0.904000,0.973786,0.760347,0.849286,0.878892,9
6,LGBM | raw + graph:degree,0.948369,0.976255,0.906540,0.053165,NaN,0.904251,0.974169,0.764558,0.851778,0.878980,9
7,LGBM | stack2 [3 seeds],0.947795,0.975721,0.905905,0.050842,0.005435,0.904674,0.956353,0.783191,0.856353,0.878042,532


### Findings
* The gain comes from the neighbour aggregates, not from the stage-1 score alone: without them the stack scores dev 0.9453 / gap 0.8961, with them 0.9538 / 0.9173 (single seed, no masking).
* With masking (stack2) the single-seed score is dev 0.9516 / gap 0.9138. Over 3 seeds stack2 scores dev 0.9478 / gap 0.9059 against 0.9448 / 0.8979 for plain LightGBM: about +0.003 dev and +0.008 gap. This is a small gain, and it is concentrated in one fold (gap fold 1, +0.031 in the single-seed paired comparison above).
* Stack2 is the final model. XGBoost over 3 seeds (0.9447 / 0.8987) is no better than LightGBM.

### 8.2 Is there a cross-step scale problem in the scores?
ROC-AUC is pooled over all test rows, so a model whose scores run higher in some time steps could lose AUC even if every step is ranked well. The check below compares the pooled AUC with the average per-step AUC, and with the pooled AUC after ranking each step separately.

**Result:** ranking each step separately made the score **worse in all five folds** (for example roll fold 0: 0.994 to 0.976, gap fold 0: 0.941 to 0.912), because the raw scores carry real information about which steps contain more illicit activity. There is no consistent gap between pooled and per-step AUC either. Per-step ranking is not used.

In [24]:
def step_diag(name):
    rows = []
    for s in ("roll", "gap"):
        for k, f in enumerate(DEV_FOLDS[s]):
            _, va = fold_pos(f); y = Y[va].astype(int); sc = np.asarray(PRED[name][(s, k)], float); st = T[va]
            per, wt = [], []
            rn = np.zeros_like(sc)
            for t in np.unique(st):
                m = st == t
                rn[m] = rankdata(sc[m]) / m.sum()
                if y[m].min() != y[m].max():
                    per.append(roc_auc_score(y[m], sc[m])); wt.append(m.sum())
            rows.append(dict(set=s, fold=k, pooled=roc_auc_score(y, sc),
                             mean_per_step=np.average(per, weights=wt),
                             pooled_after_step_rank=roc_auc_score(y, rn)))
    print(pd.DataFrame(rows).round(4))

step_diag("LGBM | raw [3 seeds]")
step_diag("LGBM | stack2 [3 seeds]")

    set  fold  pooled  mean_per_step  pooled_after_step_rank
0  roll     0  0.9940         0.9937                  0.9759
1  roll     1  0.9386         0.9308                  0.9168
2  roll     2  0.9974         0.9968                  0.9848
3   gap     0  0.9407         0.9301                  0.9118
4   gap     1  0.8560         0.8731                  0.8476
    set  fold  pooled  mean_per_step  pooled_after_step_rank
0  roll     0  0.9950         0.9948                  0.9772
1  roll     1  0.9364         0.9276                  0.9141
2  roll     2  0.9972         0.9965                  0.9846
3   gap     0  0.9396         0.9296                  0.9107
4   gap     1  0.8710         0.8823                  0.8580


### 8.3 Does number of trees matter?

To sanity check the use of the boolean variable QUICK

In [25]:
SIM_VAL_MASK = 0.25
DEV_FOLDS.setdefault("gapx", [(9, 16, 21), (15, 22, 27), (21, 28, 30)])
NS = (100, 200, 300, 500, 800)
RUN_STACK2_CURVE = True            # set False to test plain LightGBM only (much faster)
_BUF = {}

class CurveLGBM:
    # Same settings as make_lgbm, but 800 trees; remembers predictions at every n in NS.
    def __init__(self, seed):
        self.m = lgb.LGBMClassifier(n_estimators=max(NS), learning_rate=0.05, num_leaves=31, subsample=0.8,
                                    subsample_freq=1, colsample_bytree=0.8, random_state=seed, n_jobs=-1, verbose=-1)
    def fit(self, X, y, **kw):
        self.m.fit(X, y, **kw); return self
    def predict_proba(self, X):
        for n in NS:
            _BUF[n] = self.m.predict_proba(X, num_iteration=n)[:, 1]
        return self.m.predict_proba(X, num_iteration=300)

def curve_factory(seed, spw, weighted):
    return CurveLGBM(seed)

def tree_curve(label, fp):
    rows = []
    for s in ("roll", "gap", "gapx"):
        for k, f in enumerate(DEV_FOLDS[s]):
            tr, va = fold_pos(f)
            y = Y[va].astype(int)
            per_n = {n: [] for n in NS}
            for seed in (0, 1, 2):
                fp(tr, va, seed)
                for n in NS:
                    per_n[n].append(roc_auc_score(y, _BUF[n]))
            for n in NS:
                rows.append(dict(model=label, set=s, fold=k, n=n, auc=np.mean(per_n[n])))
    return pd.DataFrame(rows)

def summarize_curve(df):
    p = df.pivot_table(index=["set", "fold"], columns="n", values="auc")
    out = pd.DataFrame({
        "dev (roll+gap)": p.loc[["roll", "gap"]].mean(),
        "gap": p.loc["gap"].mean(),
        "gapx": p.loc["gapx"].mean(),
        "all 8 folds": p.mean(),
        "folds beating n=300": (p.sub(p[300], axis=0) > 0).sum(),
        "mean diff vs 300": p.sub(p[300], axis=0).mean(),
    })
    print(out.round(4).to_string())
    return p

t0 = time.time()
curve_plain = tree_curve("plain LGBM", gbdt_fp(curve_factory, X_BASE, decay=DECAY))
print("plain LightGBM:", round(time.time() - t0), "s")
p_plain = summarize_curve(curve_plain)

if RUN_STACK2_CURVE:
    t0 = time.time()
    curve_s2 = tree_curve("stack2", stacked_fp2(tree_stage1(make_hgb, X_BASE), X_BASE, curve_factory, decay=DECAY, p_mask=0.25))
    print("stack2:", round(time.time() - t0), "s")
    p_s2 = summarize_curve(curve_s2)

plain LightGBM: 79 s
     dev (roll+gap)     gap    gapx  all 8 folds  folds beating n=300  mean diff vs 300
n                                                                                      
100          0.9368  0.8875  0.9048       0.9248                    0           -0.0080
200          0.9472  0.9038  0.9183       0.9364                    5            0.0036
300          0.9448  0.8979  0.9126       0.9328                    0            0.0000
500          0.9450  0.8960  0.9079       0.9311                    4           -0.0017
800          0.9423  0.8909  0.9036       0.9278                    2           -0.0050
stack2: 198 s
     dev (roll+gap)     gap    gapx  all 8 folds  folds beating n=300  mean diff vs 300
n                                                                                      
100          0.9349  0.8832  0.8868       0.9169                    0           -0.0166
200          0.9491  0.9111  0.9162       0.9368                    3            0.00

### Findings: number of trees
Mean ROC-AUC over all 8 folds (3 seeds; the 5 dev folds plus the 3 `gapx` folds), by number of trees:

| Trees | Plain LightGBM | Stack2 |
|---|---|---|
| 100 | 0.9248 | 0.9169 |
| 200 | 0.9364 | 0.9368 |
| **300 (used)** | 0.9328 | 0.9335 |
| 500 | 0.9311 | 0.9304 |
| 800 | 0.9278 | 0.9286 |

* 100 trees is too few (-0.008 and -0.017 against 300 trees). From 200 trees on the curve is flat and then slowly falls, most of all on the three extra gap folds (stack2: 0.9162 at 200 trees, 0.8976 at 800).
* 200 trees is +0.0036 (plain) and +0.0033 (stack2) better than 300, but it wins only 5 and 3 of the 8 folds, so it does not pass the confirmation rule.
* More trees therefore do not help: `QUICK = False` (400 trees) would be expected to score the same or slightly lower, and the submitted 300-tree setting is kept.

## 9. Ensembles of tree models

In [26]:
def rank_avg(preds):
    """Blend several models' scores for the SAME rows.
    Each model's scores are converted to ranks (0 to 1) first, then averaged.
    Ranking puts models on one scale: a probability of 0.9 from one model and 0.6 from another are not comparable,
    but 'this row is in the top 5%' is. ROC-AUC depends only on the ranking, so nothing is lost."""
    return np.mean([rankdata(p) / len(p) for p in preds], axis=0)

def ensemble_dev(names, sets=("roll", "gap")):
    """ROC-AUC of a rank-average blend of previously evaluated models, on the dev folds.
    `names` are model names already evaluated (their predictions are stored in PRED), so nothing is retrained.
    Returns the mean over the roll folds and over the gap folds."""
    out = {}
    for s in sets:
        vals = []
        for k, f in enumerate(DEV_FOLDS[s]):
            _, va = fold_pos(f); y = Y[va].astype(int)
            vals.append(roc_auc_score(y, rank_avg([PRED[n][(s, k)] for n in names])))
        out[s] = float(np.mean(vals))
    return out

In [27]:
names3 = ["LGBM | raw [3 seeds]", "XGB | raw [3 seeds]", "LGBM | stack2 [3 seeds]"]
for n in names3:
    print(n, ensemble_dev([n]))
print("LGBM + XGB   :", ensemble_dev(names3[:2]))
print("LGBM + stack2:", ensemble_dev([names3[0], names3[2]]))
print("all three    :", ensemble_dev(names3))

LGBM | raw [3 seeds] {'roll': 0.976687173598935, 'gap': 0.8983290389504632}
XGB | raw [3 seeds] {'roll': 0.975791845480909, 'gap': 0.8987885575833932}
LGBM | stack2 [3 seeds] {'roll': 0.9762183191502029, 'gap': 0.905337910946374}
LGBM + XGB   : {'roll': 0.9766710290958786, 'gap': 0.8992168134572904}
LGBM + stack2: {'roll': 0.9767538397082477, 'gap': 0.9021327295111312}
all three    : {'roll': 0.9768252172550786, 'gap': 0.9016203473715241}


In [28]:
base = ["LGBM | stack2 [3 seeds]"]
for n in PRED:
    if n in base or ("gap", 0) not in PRED[n]:
        continue
    r = ensemble_dev(base + [n])
    print(f"{n:50s} gap={r['gap']:.4f} roll={r['roll']:.4f}")

LR | raw                                           gap=0.8929 roll=0.9602
RF | raw                                           gap=0.8852 roll=0.9651
HGB | raw                                          gap=0.8915 roll=0.9774
XGB | raw                                          gap=0.9053 roll=0.9769
LGBM | raw                                         gap=0.9057 roll=0.9777
LR | raw | class-weighted                          gap=0.8951 roll=0.9623
RF | raw | class-weighted                          gap=0.8847 roll=0.9691
XGB | raw | class-weighted                         gap=0.8943 roll=0.9758
LGBM | raw | class-weighted                        gap=0.8913 roll=0.9786
LGBM | rank-normalised                             gap=0.9054 roll=0.9724
LGBM | raw | decay0.95                             gap=0.9085 roll=0.9776
LGBM | rank | decay0.95                            gap=0.9061 roll=0.9715
LGBM | raw | decay0.9                              gap=0.9089 roll=0.9766
LGBM | rank | decay0.9                

### Findings
* LightGBM + XGBoost: gap 0.8992 (LightGBM alone 0.8983). All three models together: 0.9016, below stack2 alone (0.9053).
* Adding any single stored tree model to stack2 changed the gap by at most +0.004 (best: a decay-0.9 LightGBM at +0.0036), within the noise. Blends with other stack variants are the same method with different seeds.
* The submitted blend (sub3, public 0.95094) landed between its two members (sub1 0.94894, sub2 0.95157), as expected for two highly correlated models (Spearman 0.9475).

## 10. Further experiments
Sections 10.1 to 10.7 were tested against the plain LightGBM or stack2 reference and rejected; section 10.8 (per-time-step SMOTE) was adopted. A candidate is only worth a submission if it gains at least 0.005 on the gap folds with dev not lower, and then wins at least 7 of 8 folds (the 5 dev folds plus 3 extra gap folds, `gapx`). SMOTE x4 won 6 of 8 and so missed this rule by one fold, but was submitted anyway and was the best public score.

### 10.1 LightGBM settings and feature subsets
No Optuna search was run: with five validation folds, a large search mostly finds settings that fit those folds. Instead ten hand-picked configurations were tested over 3 seeds.

In [29]:
import lightgbm as lgb
def make_lgbm_cfg(**kw):
    """Make a LightGBM factory with default settings, overridden by whatever is passed in kw.
    Same (seed, spw, weighted) interface as the other factories, so it works with gbdt_fp / evaluate."""
    def factory(seed, spw, weighted):
        p = dict(n_estimators=400, learning_rate=0.05, num_leaves=31, subsample=0.8, subsample_freq=1,
                 colsample_bytree=0.8, min_child_samples=20, reg_lambda=1.0,
                 random_state=seed, n_jobs=-1, verbose=-1)
        p.update(kw)
        return lgb.LGBMClassifier(**p)
    return factory

# Five hand-picked variants, each changing one idea
CFGS = {
    "colsample.4":   dict(colsample_bytree=0.4), # each tree sees only 40% of columns (more diverse trees)
    "reg":           dict(colsample_bytree=0.4, num_leaves=15, min_child_samples=50, reg_lambda=5), # simpler trees + stronger penalty
    "slow":          dict(learning_rate=0.02, n_estimators=1000, colsample_bytree=0.5), # smaller steps, more trees
    "extra_trees":   dict(extra_trees=True, colsample_bytree=0.5, n_estimators=600), # random split points (more randomness)
    "shallow":       dict(num_leaves=7, n_estimators=600, colsample_bytree=0.5), # many small trees
}
for n, kw in CFGS.items():
    # 3 seeds each, because a single seed is too noisy (seed std about 0.005) to compare settings
    evaluate(f"LGBM cfg {n} [3 seeds]", gbdt_fp(make_lgbm_cfg(**kw), X_BASE, decay=DECAY), seeds=(0, 1, 2))
leaderboard(10)

LGBM cfg colsample.4 [3 seeds]                   dev=0.9388 gap=0.8808 roll=0.9774 fold_std=0.0650 pr=0.9011 (29s) | d_gap=-0.0240 d_dev=-0.0102
LGBM cfg reg [3 seeds]                           dev=0.9381 gap=0.8827 roll=0.9750 fold_std=0.0599 pr=0.8998 (19s) | d_gap=-0.0221 d_dev=-0.0109
LGBM cfg slow [3 seeds]                          dev=0.9411 gap=0.8840 roll=0.9791 fold_std=0.0627 pr=0.9021 (70s) | d_gap=-0.0209 d_dev=-0.0079
LGBM cfg extra_trees [3 seeds]                   dev=0.9256 gap=0.8591 roll=0.9700 fold_std=0.0789 pr=0.8966 (35s) | d_gap=-0.0457 d_dev=-0.0233
LGBM cfg shallow [3 seeds]                       dev=0.9448 gap=0.9003 roll=0.9745 fold_std=0.0515 pr=0.9027 (21s) | d_gap=-0.0045 d_dev=-0.0041


,model,roc_dev,roc_roll,roc_gap,fold_std,seed_std,pr_dev,prec,rec,f1,f1_best,secs
0,LGBM | stack: HGB stage1 + nbr preds,0.953790,0.978117,0.917301,0.043487,NaN,0.909398,0.959326,0.792121,0.862043,0.884128,32
1,"LGBM | stack2: no train mask, 25% val masked",0.952038,0.977356,0.914061,0.044023,NaN,0.906626,0.967312,0.771657,0.853329,0.877135,34
2,"LGBM | stack2: train mask .25, 25% val masked",0.951575,0.976782,0.913765,0.044375,NaN,0.907744,0.962371,0.779989,0.856398,0.880966,35
3,LGBM | raw | decay0.9,0.949634,0.975765,0.910439,0.047889,NaN,0.905858,0.978196,0.764077,0.853367,0.879845,9
4,LGBM | raw | decay0.95,0.950830,0.977837,0.910319,0.048825,NaN,0.905475,0.978652,0.762078,0.851528,0.879394,9
5,LGBM | raw + graph:nbr_structure,0.947782,0.973860,0.908664,0.050991,NaN,0.904000,0.973786,0.760347,0.849286,0.878892,9
6,LGBM | raw + graph:degree,0.948369,0.976255,0.906540,0.053165,NaN,0.904251,0.974169,0.764558,0.851778,0.878980,9
7,LGBM | stack2 [3 seeds],0.947795,0.975721,0.905905,0.050842,0.005435,0.904674,0.956353,0.783191,0.856353,0.878042,532
8,LGBM | raw,0.948957,0.978375,0.904829,0.054624,NaN,0.904786,0.977424,0.759984,0.850067,0.879476,9
9,LGBM | raw | drop top20 drift feats,0.948778,0.978865,0.903648,0.050295,NaN,0.904969,0.982422,0.768435,0.856871,0.875953,8


In [30]:
K = 94 # number of local features (not including the aggregated neighbour features)
num = np.array([int(c.split("_")[1]) for c in feat])
local_idx = np.where(num < K)[0]
evaluate("LGBM | local feats only [3 seeds]", gbdt_fp(MAIN_FACTORY, X_BASE[:, local_idx], decay=DECAY), seeds=(0, 1, 2))

LGBM | local feats only [3 seeds]                dev=0.9422 gap=0.8915 roll=0.9760 fold_std=0.0572 pr=0.8969 (22s) | d_gap=-0.0133 d_dev=-0.0067


{'model': 'LGBM | local feats only [3 seeds]',
 'roc_dev': np.float64(0.9422181850092943),
 'roc_roll': np.float64(0.976030215331705),
 'roc_gap': np.float64(0.8915001395256783),
 'fold_std': np.float64(0.057171478970604546),
 'seed_std': np.float64(0.0034525595410871282),
 'pr_dev': np.float64(0.896899699464585),
 'prec': np.float64(0.9814621381185749),
 'rec': np.float64(0.713393533803424),
 'f1': np.float64(0.8210160109741877),
 'f1_best': np.float64(0.8639758767449057),
 'secs': 22}

In [31]:
CFGS2 = {
    "colsample1.0":        dict(colsample_bytree=1.0),
    "shallow_col1.0":      dict(num_leaves=7, n_estimators=600, colsample_bytree=1.0),
    "leaves63":            dict(num_leaves=63, colsample_bytree=1.0),
    "shallow_slow_col1.0": dict(num_leaves=7, learning_rate=0.03, n_estimators=1000, colsample_bytree=1.0),
}
for n, kw in CFGS2.items():
    evaluate(f"LGBM cfg {n} [3 seeds]", gbdt_fp(make_lgbm_cfg(**kw), X_BASE, decay=DECAY), seeds=(0, 1, 2))
leaderboard(12)

LGBM cfg colsample1.0 [3 seeds]                  dev=0.9408 gap=0.8862 roll=0.9772 fold_std=0.0612 pr=0.9009 (39s) | d_gap=-0.0186 d_dev=-0.0082
LGBM cfg shallow_col1.0 [3 seeds]                dev=0.9461 gap=0.9019 roll=0.9755 fold_std=0.0495 pr=0.9029 (30s) | d_gap=-0.0029 d_dev=-0.0029
LGBM cfg leaves63 [3 seeds]                      dev=0.9381 gap=0.8780 roll=0.9781 fold_std=0.0676 pr=0.8993 (48s) | d_gap=-0.0268 d_dev=-0.0109
LGBM cfg shallow_slow_col1.0 [3 seeds]           dev=0.9449 gap=0.8989 roll=0.9756 fold_std=0.0514 pr=0.9023 (49s) | d_gap=-0.0059 d_dev=-0.0040


,model,roc_dev,roc_roll,roc_gap,fold_std,seed_std,pr_dev,prec,rec,f1,f1_best,secs
0,LGBM | stack: HGB stage1 + nbr preds,0.953790,0.978117,0.917301,0.043487,NaN,0.909398,0.959326,0.792121,0.862043,0.884128,32
1,"LGBM | stack2: no train mask, 25% val masked",0.952038,0.977356,0.914061,0.044023,NaN,0.906626,0.967312,0.771657,0.853329,0.877135,34
2,"LGBM | stack2: train mask .25, 25% val masked",0.951575,0.976782,0.913765,0.044375,NaN,0.907744,0.962371,0.779989,0.856398,0.880966,35
3,LGBM | raw | decay0.9,0.949634,0.975765,0.910439,0.047889,NaN,0.905858,0.978196,0.764077,0.853367,0.879845,9
4,LGBM | raw | decay0.95,0.950830,0.977837,0.910319,0.048825,NaN,0.905475,0.978652,0.762078,0.851528,0.879394,9
5,LGBM | raw + graph:nbr_structure,0.947782,0.973860,0.908664,0.050991,NaN,0.904000,0.973786,0.760347,0.849286,0.878892,9
6,LGBM | raw + graph:degree,0.948369,0.976255,0.906540,0.053165,NaN,0.904251,0.974169,0.764558,0.851778,0.878980,9
7,LGBM | stack2 [3 seeds],0.947795,0.975721,0.905905,0.050842,0.005435,0.904674,0.956353,0.783191,0.856353,0.878042,532
8,LGBM | raw,0.948957,0.978375,0.904829,0.054624,NaN,0.904786,0.977424,0.759984,0.850067,0.879476,9
9,LGBM | raw | drop top20 drift feats,0.948778,0.978865,0.903648,0.050295,NaN,0.904969,0.982422,0.768435,0.856871,0.875953,8


### Findings
Reference: plain LightGBM over 3 seeds, dev 0.9448 / gap 0.8979.

| Configuration | Dev | Gap |
|---|---|---|
| column sampling 0.4 | 0.9388 | 0.8808 |
| strong regularisation | 0.9381 | 0.8827 |
| slow learning rate | 0.9411 | 0.8840 |
| extra trees | 0.9256 | 0.8591 |
| shallow (7 leaves), column sampling 0.5 | 0.9448 | 0.9003 |
| column sampling 1.0 | 0.9408 | 0.8862 |
| shallow, column sampling 1.0 | 0.9461 | 0.9019 |
| 63 leaves | 0.9381 | 0.8780 |
| shallow, slow, column sampling 1.0 | 0.9449 | 0.8989 |
| local features only (no aggregated block) | 0.9422 | 0.8915 |

### 10.2 Stack2 with a shallow final model
The shallow configuration was combined with stack2 (dev 0.9511 / gap 0.9128, +0.007 gap over stack2) and checked on the frozen holdout (ROC 0.9970, the same as stack2; PR-AUC 0.9885 and best-threshold F1 0.9634, slightly below stack2's 0.9891 and 0.9685). It was written out as `sub4` (see the appendix) and **scored 0.94725 on the leaderboard, below sub1 and sub2**. The shallow setting had been chosen after seeing the same gap folds, so the apparent gain was selection noise. This is the reason for the stricter 8-fold confirmation rule above.

In [32]:
# 1) blend, no refit
print(ensemble_dev(["LGBM | stack2 [3 seeds]", "LGBM cfg shallow_col1.0 [3 seeds]"]))

# 2) stack2 with the shallow final model
SHALLOW = make_lgbm_cfg(num_leaves=7, n_estimators=600, colsample_bytree=1.0)
evaluate("LGBM | stack2 + shallow final [3 seeds]",
         stacked_fp2(tree_stage1(make_hgb, X_BASE), X_BASE, SHALLOW, decay=DECAY, p_mask=0.25), seeds=(0, 1, 2))

{'roll': 0.977264397697272, 'gap': 0.9057704050935802}
LGBM | stack2 + shallow final [3 seeds]          dev=0.9511 gap=0.9128 roll=0.9767 fold_std=0.0424 pr=0.9074 (104s) | d_gap=+0.0080 d_dev=+0.0022


{'model': 'LGBM | stack2 + shallow final [3 seeds]',
 'roc_dev': np.float64(0.9511395085510047),
 'roc_roll': np.float64(0.9767021319893638),
 'roc_gap': np.float64(0.9127955733934662),
 'fold_std': np.float64(0.042449441836785264),
 'seed_std': np.float64(0.004250476041181224),
 'pr_dev': np.float64(0.9074047721353379),
 'prec': np.float64(0.9326342500347338),
 'rec': np.float64(0.8060282017539742),
 'f1': np.float64(0.8587729907798689),
 'f1_best': np.float64(0.8788200489348451),
 'secs': 104}

In [33]:
print("SIM_VAL_MASK is currently:", SIM_VAL_MASK)   # should print 0.25

SHALLOW = make_lgbm_cfg(num_leaves=7, n_estimators=600, colsample_bytree=1.0)
fp_new = stacked_fp2(tree_stage1(make_hgb, X_BASE), X_BASE, SHALLOW, decay=DECAY, p_mask=0.25)
holdout_eval("LGBM | stack2 + shallow final", fp_new, seeds=(0, 1, 2))

SIM_VAL_MASK is currently: 0.25
[HOLDOUT] LGBM | stack2 + shallow final                roc=0.9970 pr=0.9885 f1_best=0.9634


{'model': 'LGBM | stack2 + shallow final',
 'roc_auc': np.float64(0.9969715188193448),
 'pr_auc': np.float64(0.9885149273186282),
 'prec': 0.953757225433526,
 'rec': 0.9565217391304348,
 'f1': 0.9551374819102749,
 'f1_best': 0.9633967789160446}

### 10.3 Self-training on the target period (pseudo-labels)
Fit on the labelled rows, score the rows of the period to be predicted (the validation window, or the test steps in the final fit, plus unlabelled rows in that window), keep only very confident predictions as pseudo-labels, and refit. It never uses validation labels. The pseudo-labelled rows get a lower weight.

**Result:** dev 0.9246 / gap 0.8649 (about -0.04 on gap). Rejected.

In [34]:
def selftrain_fp(factory, X, hi=0.9, lo=0.01, w_pseudo=0.5, decay=None):
    def fp(tr, va, seed):
        y = Y[tr].astype(int)
        w = np.ones(len(tr)) if decay is None else decay ** (T[tr].max() - T[tr])
        m = fit_model(factory, X[tr], y, w, seed, False)
        t_lo, t_hi = T[va].min(), T[va].max()
        pool = np.unique(np.concatenate([va, UNK[(T[UNK] >= t_lo) & (T[UNK] <= t_hi)]]))
        p = m.predict_proba(X[pool])[:, 1]
        keep = (p >= hi) | (p <= lo)
        pl = pool[keep]; yl = (p[keep] >= hi).astype(int)
        X2 = np.vstack([X[tr], X[pl]]); y2 = np.concatenate([y, yl])
        w2 = np.concatenate([w, np.full(len(pl), w_pseudo)])
        m2 = fit_model(factory, X2, y2, w2, seed, False)
        return m2.predict_proba(X[va])[:, 1]
    return fp

evaluate(f"{MAIN} | self-training", selftrain_fp(MAIN_FACTORY, X_BASE, decay=DECAY))
leaderboard(12)

LGBM | self-training                             dev=0.9246 gap=0.8649 roll=0.9644 fold_std=0.0725 pr=0.8938 (27s) | d_gap=-0.0399 d_dev=-0.0243


,model,roc_dev,roc_roll,roc_gap,fold_std,seed_std,pr_dev,prec,rec,f1,f1_best,secs
0,LGBM | stack: HGB stage1 + nbr preds,0.953790,0.978117,0.917301,0.043487,NaN,0.909398,0.959326,0.792121,0.862043,0.884128,32
1,"LGBM | stack2: no train mask, 25% val masked",0.952038,0.977356,0.914061,0.044023,NaN,0.906626,0.967312,0.771657,0.853329,0.877135,34
2,"LGBM | stack2: train mask .25, 25% val masked",0.951575,0.976782,0.913765,0.044375,NaN,0.907744,0.962371,0.779989,0.856398,0.880966,35
3,LGBM | stack2 + shallow final [3 seeds],0.951140,0.976702,0.912796,0.042449,0.004250,0.907405,0.932634,0.806028,0.858773,0.878820,104
4,LGBM | raw | decay0.9,0.949634,0.975765,0.910439,0.047889,NaN,0.905858,0.978196,0.764077,0.853367,0.879845,9
5,LGBM | raw | decay0.95,0.950830,0.977837,0.910319,0.048825,NaN,0.905475,0.978652,0.762078,0.851528,0.879394,9
6,LGBM | raw + graph:nbr_structure,0.947782,0.973860,0.908664,0.050991,NaN,0.904000,0.973786,0.760347,0.849286,0.878892,9
7,LGBM | raw + graph:degree,0.948369,0.976255,0.906540,0.053165,NaN,0.904251,0.974169,0.764558,0.851778,0.878980,9
8,LGBM | stack2 [3 seeds],0.947795,0.975721,0.905905,0.050842,0.005435,0.904674,0.956353,0.783191,0.856353,0.878042,532
9,LGBM | raw,0.948957,0.978375,0.904829,0.054624,NaN,0.904786,0.977424,0.759984,0.850067,0.879476,9


### 10.4 Richer neighbour features ("stack3")
Four additions to stack2, tested one at a time and together: richer neighbour summaries (min, std, share of neighbours above 0.5, 3-hop mean), connected-component features (mean and max stage-1 score over the component, component size), PageRank-style propagation of the stage-1 scores (`R = aPR + (1-a)e`), a more diverse stage 1 (HGB + LightGBM + XGBoost), and a second stacking round.

Reference: stack2 over 3 seeds, dev 0.9478 / gap 0.9059.

In [35]:
from scipy.sparse.csgraph import connected_components
COMP_USE_SIZE = True
# components
_, COMP = connected_components(A_und, directed=False)
COMP_N = np.bincount(COMP)
COMP_SIZE = COMP_N[COMP].astype(float)
print("components:", len(COMP_N), "| largest:", int(COMP_N.max()),
      "| share of rows in components of size >= 10:", round(float((COMP_SIZE >= 10).mean()), 3))

# row-normalised adjacency for PageRank-style propagation
_d = np.asarray(A_und.sum(1)).ravel().astype(np.float64)
Pn = sp.diags(np.divide(1.0, _d, out=np.zeros_like(_d), where=_d > 0)) @ A_und

def graph_feats(p, rich=True, comp=True, prop=True):
    p = np.asarray(p, dtype=np.float64)
    cols = [nbr_aggregate(p)]                                
    if rich:
        m_und = mean_over(A_und, p)
        std = np.sqrt(np.maximum(mean_over(A_und, p * p) - m_und ** 2, 0))
        mn = -row_max(A_und, -p, fill=-1.0)
        share = mean_over(A_und, (p > 0.5).astype(np.float64))
        m2 = mean_over(A_und, np.where(m_und >= 0, m_und, 0.0))
        m3 = mean_over(A_und, np.where(m2 >= 0, m2, 0.0))
        cols.append(np.column_stack([mn, std, share, m3]))
    if comp:
        cmean = (np.bincount(COMP, weights=p) / COMP_N)[COMP]
        cmax = np.full(len(COMP_N), -1.0); np.maximum.at(cmax, COMP, p); cmax = cmax[COMP]
        cols.append(np.column_stack([cmean, cmax, COMP_SIZE]) if COMP_USE_SIZE else np.column_stack([cmean, cmax]))
    if prop:
        for a in (0.5, 0.9):                                    # R = a*P*R + (1-a)*e, e = predicted scores
            r = p.copy()
            for _ in range(15):
                r = a * (Pn @ r) + (1 - a) * p
            cols.append(r[:, None])
    return np.hstack(cols)

def multi_stage1(factories, X):
    def f(fit, seed):
        y = Y[fit].astype(int)
        ps = [fit_model(fac, X[fit], y, None, seed, False).predict_proba(X)[:, 1] for fac in factories]
        return np.mean(ps, axis=0)[:, None]
    return f

def _stage2_pred(factory, X, fit, seed):
    m = fit_model(factory, X[fit], Y[fit].astype(int), None, seed, False)
    return m.predict_proba(X)[:, 1:2]

def stacked_fp3(stage1, X_base, final_factory, decay=None, k=4, p_mask=0.25,
                rich=True, comp=True, prop=True, rounds=1):
    def fp(tr, va, seed):
        S = oof_scores(lambda fit: stage1(fit, seed), tr, k)
        rng = np.random.RandomState(seed)
        m = np.zeros(N, bool)
        m[tr] = rng.rand(len(tr)) < p_mask
        m[va] = rng.rand(len(va)) < SIM_VAL_MASK
        iso = und_deg == 0
        def build_G(score):
            G = graph_feats(score, rich, comp, prop).astype(np.float64)
            G[iso | m] = np.nan
            return G
        X = np.hstack([X_base, S, build_G(S[:, 0])]).astype(np.float32)
        if rounds == 2:                                        # second round: block-OOF stage-2 scores
            S2 = oof_scores(lambda fit: _stage2_pred(final_factory, X, fit, seed), tr, k)
            X = np.hstack([X, S2, build_G(S2[:, 0])]).astype(np.float32)
        return gbdt_fp(final_factory, X, False, decay)(tr, va, seed)
    return fp

components: 4489 | largest: 7880 | share of rows in components of size >= 10: 0.966


In [36]:
SIM_VAL_MASK = 0.25
assert SIM_VAL_MASK == 0.25, "set SIM_VAL_MASK = 0.25 first"
S1_HGB = tree_stage1(make_hgb, X_BASE)

def run3(tag, stage1=S1_HGB, **kw):
    return evaluate(f"LGBM | stack3 {tag} [3 seeds]",
                    stacked_fp3(stage1, X_BASE, MAIN_FACTORY, decay=DECAY, **kw), seeds=(0, 1, 2))

run3("rich", rich=True,  comp=False, prop=False)
run3("comp", rich=False, comp=True,  prop=False)
run3("prop", rich=False, comp=False, prop=True)
run3("all",  rich=True,  comp=True,  prop=True)
print("reference: stack2 [3 seeds] dev=0.9478 gap=0.9059")

LGBM | stack3 rich [3 seeds]                     dev=0.9472 gap=0.9046 roll=0.9756 fold_std=0.0503 pr=0.9050 (110s) | d_gap=-0.0002 d_dev=-0.0018
LGBM | stack3 comp [3 seeds]                     dev=0.9519 gap=0.9140 roll=0.9772 fold_std=0.0457 pr=0.9070 (104s) | d_gap=+0.0092 d_dev=+0.0030
LGBM | stack3 prop [3 seeds]                     dev=0.9479 gap=0.9093 roll=0.9736 fold_std=0.0488 pr=0.9055 (108s) | d_gap=+0.0045 d_dev=-0.0011
LGBM | stack3 all [3 seeds]                      dev=0.9462 gap=0.9040 roll=0.9743 fold_std=0.0511 pr=0.9046 (107s) | d_gap=-0.0009 d_dev=-0.0028
reference: stack2 [3 seeds] dev=0.9478 gap=0.9059


In [37]:
SIM_VAL_MASK = 0.25
for nm, idx in [("train labelled", LAB), ("train unknown", UNK), ("test", TEST_POS)]:
    print(f"{nm:15s} comp size p10/p50/p90/p99:", np.percentile(COMP_SIZE[idx], [10, 50, 90, 99]).round(0))
COMP_USE_SIZE = False
run3("comp no size", rich=False, comp=True, prop=False)
COMP_USE_SIZE = True

train labelled  comp size p10/p50/p90/p99: [2486. 4528. 6803. 7880.]
train unknown   comp size p10/p50/p90/p99: [2486. 4525. 6727. 7880.]
test            comp size p10/p50/p90/p99: [  1. 485. 940. 941.]
LGBM | stack3 comp no size [3 seeds]             dev=0.9492 gap=0.9100 roll=0.9753 fold_std=0.0470 pr=0.9056 (122s) | d_gap=+0.0052 d_dev=+0.0002


In [38]:
DEV_FOLDS["gapx"] = [(9, 16, 21), (15, 22, 27), (21, 28, 30)]     

def evalx(name, fp):
    evaluate(name + " [gapx]", fp, seeds=(0, 1, 2), sets=("gapx",))

def fold_aucs(name):
    out = {}
    for s in ("roll", "gap"):
        for k, f in enumerate(DEV_FOLDS[s]):
            _, va = fold_pos(f); out[(s, k)] = roc_auc_score(Y[va].astype(int), PRED[name][(s, k)])
    for k, f in enumerate(DEV_FOLDS["gapx"]):
        _, va = fold_pos(f); out[("gapx", k)] = roc_auc_score(Y[va].astype(int), PRED[name + " [gapx]"][("gapx", k)])
    return pd.Series(out)

def compare8(a, b):
    d = pd.DataFrame({"a": fold_aucs(a), "b": fold_aucs(b)}); d["diff"] = d.a - d.b
    print(d.round(4))
    print(f"{a!r} wins {(d['diff'] > 0).sum()} of {len(d)} folds | mean diff {d['diff'].mean():+.4f}")

CAND = "LGBM | stack3 all [3 seeds]"
evalx("LGBM | stack2 [3 seeds]",
      stacked_fp2(tree_stage1(make_hgb, X_BASE), X_BASE, MAIN_FACTORY, decay=DECAY, p_mask=0.25))
evalx(CAND, stacked_fp3(S1_HGB, X_BASE, MAIN_FACTORY, decay=DECAY, rich=True, comp=True, prop=True))  # same flags as CAND
compare8(CAND, "LGBM | stack2 [3 seeds]")

LGBM | stack2 [3 seeds] [gapx]                   dev=0.9097 gap=nan roll=nan fold_std=0.0864 pr=0.8483 (70s) | d_gap=+nan d_dev=-0.0393
LGBM | stack3 all [3 seeds] [gapx]               dev=0.9018 gap=nan roll=nan fold_std=0.0938 pr=0.8458 (69s) | d_gap=+nan d_dev=-0.0471
             a       b    diff
roll 0  0.9951  0.9950  0.0001
     1  0.9313  0.9364 -0.0051
     2  0.9976  0.9972  0.0004
gap  0  0.9356  0.9396 -0.0041
     1  0.8739  0.8710  0.0029
gapx 0  0.9913  0.9916 -0.0002
     1  0.9191  0.9255 -0.0064
     2  0.8039  0.8172 -0.0133
'LGBM | stack3 all [3 seeds]' wins 3 of 8 folds | mean diff -0.0032


In [39]:
SIM_VAL_MASK = 0.25
S1_MULTI = multi_stage1([make_hgb, MAIN_FACTORY, make_xgb], X_BASE)
run3("base + diverse stage1", stage1=S1_MULTI, rich=False, comp=False, prop=False)
run3("base + round 2", rounds=2, rich=False, comp=False, prop=False)

LGBM | stack3 base + diverse stage1 [3 seeds]    dev=0.9498 gap=0.9094 roll=0.9767 fold_std=0.0475 pr=0.9067 (477s) | d_gap=+0.0046 d_dev=+0.0009
LGBM | stack3 base + round 2 [3 seeds]           dev=0.9558 gap=0.9226 roll=0.9779 fold_std=0.0385 pr=0.9087 (277s) | d_gap=+0.0178 d_dev=+0.0068


{'model': 'LGBM | stack3 base + round 2 [3 seeds]',
 'roc_dev': np.float64(0.9558011600406209),
 'roc_roll': np.float64(0.9779320103609989),
 'roc_gap': np.float64(0.9226048845600539),
 'fold_std': np.float64(0.038493959162032035),
 'seed_std': np.float64(0.0046160702627257265),
 'pr_dev': np.float64(0.9087094707938208),
 'prec': np.float64(0.9477422664115375),
 'rec': np.float64(0.7782682507510609),
 'f1': np.float64(0.8481057287686316),
 'f1_best': np.float64(0.8747084549963897),
 'secs': 277}

In [40]:
SIM_VAL_MASK = 0.25
CAND = "LGBM | stack3 base + round 2 [3 seeds]"
fp_r2 = stacked_fp3(S1_HGB, X_BASE, MAIN_FACTORY, decay=DECAY, rounds=2, rich=False, comp=False, prop=False)
evalx(CAND, fp_r2)
compare8(CAND, "LGBM | stack2 [3 seeds]")

LGBM | stack3 base + round 2 [3 seeds] [gapx]    dev=0.9066 gap=nan roll=nan fold_std=0.0927 pr=0.8462 (137s) | d_gap=+nan d_dev=-0.0423
             a       b    diff
roll 0  0.9950  0.9950 -0.0000
     1  0.9424  0.9364  0.0060
     2  0.9977  0.9972  0.0005
gap  0  0.9372  0.9396 -0.0024
     1  0.9087  0.8710  0.0376
gapx 0  0.9898  0.9916 -0.0018
     1  0.9275  0.9255  0.0021
     2  0.8136  0.8172 -0.0036
'LGBM | stack3 base + round 2 [3 seeds]' wins 4 of 8 folds | mean diff +0.0048


### Findings
| Variant | Dev | Gap | Verdict |
|---|---|---|---|
| richer summaries | 0.9472 | 0.9046 | nothing |
| component features | 0.9519 | 0.9140 | rejected, see below |
| propagation | 0.9479 | 0.9093 | noise |
| all three | 0.9462 | 0.9040 | worse; wins 3 of 8 folds |
| component features without size | 0.9492 | 0.9100 | below the bar |
| diverse stage 1 | 0.9498 | 0.9094 | noise, and about 4-5x slower (477 s vs 104-122 s for the other stack3 variants) |
| second stacking round | 0.9558 | 0.9226 | fails the 8-fold check |

* **Component features look good only because validation looks like training.** There are 4,489 components with a largest of 7,880, and training nodes sit in components with a median size of **4,528**, essentially a whole time step. In the test the median component size is **485** (99th percentile 941), so component size and the component mean score likely mean something different on the test. Rejected.
* **The second round wins 4 of 8 folds.** Its mean advantage over stack2 is +0.0048, but one fold (gap fold 1) contributes +0.0376. On the other seven folds the mean difference is +0.0001. Rejected.

### 10.5 Unlabelled data, window length, class balance and a sparse test graph

1. **Unknown rows as weak negatives** (PU learning): add the unknown rows from the training period as label 0 with a small weight.
2. **Sliding-window training:** train only on the last 8, 12 or 16 steps before the cut-off.
3. **Balanced bagging:** average models fitted on all illicit rows and a random sample of licit rows.
4. **Edge-dropout stacking:** build the neighbour features of training rows from a graph with random edges removed (`Q = 0.421`), to match the sparse test graph. Validation rows are thinned too, so the validation looks like the test; the comparison is against stack2 evaluated under the same thinned validation.

In [41]:
# 1) unknown rows as weak negatives
def pu_fp(factory, X, w_unk=0.1, decay=None):
    def fp(tr, va, seed):
        te = T[tr].max()
        unk = UNK[T[UNK] <= te]                       # only unknown rows from the training period
        pos = np.concatenate([tr, unk])
        y = np.concatenate([Y[tr].astype(int), np.zeros(len(unk), int)])
        w = np.concatenate([np.ones(len(tr)), np.full(len(unk), w_unk)])
        return fit_model(factory, X[pos], y, w, seed, False).predict_proba(X[va])[:, 1]
    return fp

# 2) sliding-window training
def window_fp(factory, X, win=12):
    def fp(tr, va, seed):
        tr2 = tr[T[tr] > T[tr].max() - win]
        return fit_model(factory, X[tr2], Y[tr2].astype(int), None, seed, False).predict_proba(X[va])[:, 1]
    return fp

# 3) balanced bagging (under-sample licit rows)
def bag_fp(factory, X, n_bags=8, neg_ratio=3.0):
    def fp(tr, va, seed):
        y = Y[tr].astype(int); pos, neg = tr[y == 1], tr[y == 0]
        rng = np.random.RandomState(seed); n_neg = min(len(neg), int(neg_ratio * len(pos)))
        out = np.zeros(len(va))
        for b in range(n_bags):
            sel = np.concatenate([pos, rng.choice(neg, n_neg, replace=False)])
            m = fit_model(factory, X[sel], Y[sel].astype(int), None, seed * 100 + b, False)
            out += m.predict_proba(X[va])[:, 1]
        return out / n_bags
    return fp

# 4) edge-dropout stacking
def nbr_aggregate_g(p, Ad, Ai, Au):
    p = np.asarray(p, dtype=np.float64)
    m_in, m_out, m_und = mean_over(Ai, p), mean_over(Ad, p), mean_over(Au, p)
    mx = row_max(Au, p)
    m2 = mean_over(Au, np.where(m_und >= 0, m_und, 0.0))
    return np.column_stack([m_in, m_out, m_und, mx, m2])

def thin_graph(q, seed):
    rng = np.random.RandomState(seed); c = A_dir.tocoo()
    keep = rng.rand(len(c.row)) >= q
    Ad = sp.csr_matrix((np.ones(keep.sum(), np.float32), (c.row[keep], c.col[keep])), shape=(N, N))
    return Ad, Ad.T.tocsr(), ((Ad + Ad.T) > 0).astype(np.float32).tocsr()

def stacked_fp_sparse(stage1, X_base, final_factory, q_train, q_val, decay=None, k=4, p_mask=0.0):
    def fp(tr, va, seed):
        S = oof_scores(lambda fit: stage1(fit, seed), tr, k)
        p = S[:, 0]
        def agg_with(q, s):
            if q <= 0:
                A, iso = nbr_aggregate(p), (und_deg == 0)
            else:
                Ad, Ai, Au = thin_graph(q, s)
                A, iso = nbr_aggregate_g(p, Ad, Ai, Au), (np.asarray(Au.sum(1)).ravel() == 0)
            A = A.astype(np.float64); A[iso] = np.nan
            return A
        agg = agg_with(q_train, seed)
        agg[va] = agg_with(q_val, seed + 1000)[va]
        if p_mask > 0:
            agg[tr[np.random.RandomState(seed).rand(len(tr)) < p_mask]] = np.nan
        X = np.hstack([X_base, S, agg]).astype(np.float32)
        return gbdt_fp(final_factory, X, False, decay)(tr, va, seed)
    return fp

In [42]:
def run(name, fp):
    return evaluate(name + " [3 seeds]", fp, seeds=(0, 1, 2))

for w in (0.03, 0.1, 0.3):
    run(f"LGBM | PU unknown w={w}", pu_fp(MAIN_FACTORY, X_BASE, w_unk=w))
for win in (8, 12, 16):
    run(f"LGBM | window {win}", window_fp(MAIN_FACTORY, X_BASE, win=win))
for r in (1.0, 3.0):
    run(f"LGBM | bagging ratio {r}", bag_fp(MAIN_FACTORY, X_BASE, neg_ratio=r))
print("reference: LGBM | raw [3 seeds] dev=0.9448 gap=0.8979")

LGBM | PU unknown w=0.03 [3 seeds]               dev=0.9248 gap=0.8617 roll=0.9669 fold_std=0.0695 pr=0.8816 (73s) | d_gap=-0.0432 d_dev=-0.0242
LGBM | PU unknown w=0.1 [3 seeds]                dev=0.9125 gap=0.8414 roll=0.9599 fold_std=0.0841 pr=0.8761 (76s) | d_gap=-0.0634 d_dev=-0.0365
LGBM | PU unknown w=0.3 [3 seeds]                dev=0.9080 gap=0.8354 roll=0.9564 fold_std=0.0887 pr=0.8741 (72s) | d_gap=-0.0695 d_dev=-0.0410
LGBM | window 8 [3 seeds]                        dev=0.9458 gap=0.9003 roll=0.9762 fold_std=0.0552 pr=0.9006 (24s) | d_gap=-0.0045 d_dev=-0.0031
LGBM | window 12 [3 seeds]                       dev=0.9408 gap=0.8924 roll=0.9731 fold_std=0.0617 pr=0.9007 (28s) | d_gap=-0.0125 d_dev=-0.0082
LGBM | window 16 [3 seeds]                       dev=0.9441 gap=0.8991 roll=0.9741 fold_std=0.0562 pr=0.9027 (31s) | d_gap=-0.0057 d_dev=-0.0049
LGBM | bagging ratio 1.0 [3 seeds]               dev=0.9453 gap=0.8950 roll=0.9789 fold_std=0.0571 pr=0.8996 (147s) | d_gap=-0.009

In [43]:
S1_HGB = tree_stage1(make_hgb, X_BASE)
def runs(tag, **kw):
    return run(f"LGBM | sparse {tag}", stacked_fp_sparse(S1_HGB, X_BASE, MAIN_FACTORY, decay=DECAY, **kw))

runs("val-thinned, train dense, mask .25", q_train=0.0, q_val=Q, p_mask=0.25)   # current method under test-like conditions
runs("val-thinned, train thinned",         q_train=Q,   q_val=Q, p_mask=0.0)
runs("val-thinned, train thinned + mask",  q_train=Q,   q_val=Q, p_mask=0.25)

LGBM | sparse val-thinned, train dense, mask .25 [3 seeds] dev=0.9478 gap=0.9059 roll=0.9757 fold_std=0.0509 pr=0.9045 (100s) | d_gap=+0.0010 d_dev=-0.0012
LGBM | sparse val-thinned, train thinned [3 seeds] dev=0.9441 gap=0.8960 roll=0.9762 fold_std=0.0548 pr=0.9028 (97s) | d_gap=-0.0088 d_dev=-0.0049
LGBM | sparse val-thinned, train thinned + mask [3 seeds] dev=0.9471 gap=0.9050 roll=0.9751 fold_std=0.0490 pr=0.9040 (102s) | d_gap=+0.0002 d_dev=-0.0019


{'model': 'LGBM | sparse val-thinned, train thinned + mask [3 seeds]',
 'roc_dev': np.float64(0.9470810425035274),
 'roc_roll': np.float64(0.9751074113275465),
 'roc_gap': np.float64(0.9050414892674992),
 'fold_std': np.float64(0.04904892461644575),
 'seed_std': np.float64(0.003974252127879429),
 'pr_dev': np.float64(0.9040025997159926),
 'prec': np.float64(0.9586352563860606),
 'rec': np.float64(0.7804546472592029),
 'f1': np.float64(0.8554174617488097),
 'f1_best': np.float64(0.8759899337450718),
 'secs': 102}

In [12]:
def confirm(cand_name, cand_fp, ref_name, ref_fp):
    # Evaluate a candidate and its reference on the three extra gap folds, then compare all 8 folds.
    if ref_name + " [gapx]" not in PRED:
        evalx(ref_name, ref_fp)
    evalx(cand_name, cand_fp)
    compare8(cand_name, ref_name)

### Findings
Reference: plain LightGBM over 3 seeds, dev 0.9448 / gap 0.8979.

| Idea | Dev | Gap |
|---|---|---|
| unknown as negatives, weight 0.03 | 0.9248 | 0.8617 |
| unknown as negatives, weight 0.1 | 0.9125 | 0.8414 |
| unknown as negatives, weight 0.3 | 0.9080 | 0.8354 |
| window 8 / 12 / 16 steps | 0.9458 / 0.9408 / 0.9441 | 0.9003 / 0.8924 / 0.8991 |
| balanced bagging 1:1 / 3:1 | 0.9453 / 0.9460 | 0.8950 / 0.8966 |

* Unknown rows are not safe negatives: performance falls steadily as their weight rises, so they contain many illicit-like transactions.
* Window length and bagging make no consistent difference.
* Under a thinned validation graph, stack2 keeps its edge over plain LightGBM (gap 0.9059 vs 0.8979). Training on a matching thinned graph does not help (gap 0.8960, or 0.9050 together with the row masking that stack2 already uses).

### 10.6 Graph neural networks
Full-batch GCN and GraphSAGE on the whole graph, loss on the labelled training nodes only.

In [ ]:
import torch, torch.nn as nn
from torch_geometric.nn import GATConv, GCNConv, SAGEConv, DeepGraphInfomax
GDEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("torch device:", GDEV)

mu, sd = X_raw.mean(0), X_raw.std(0) + 1e-6                       # label-free scaling
X_STD = np.clip((X_raw - mu) / sd, -5, 5).astype(np.float32)
GX = {"std": torch.tensor(X_STD, device=GDEV),
      "rank": torch.tensor(X_rank, dtype=torch.float32, device=GDEV)}
_c = A_und.tocoo()
GEI = torch.tensor(np.vstack([_c.row, _c.col]), dtype=torch.long, device=GDEV)

class GNN2(nn.Module):
    def __init__(self, kind, d_in, hid=64, heads=4, drop=0.3):
        super().__init__()
        self.drop = drop
        if kind == "gat":
            self.c1 = GATConv(d_in, hid, heads=heads, dropout=drop)
            self.c2 = GATConv(hid * heads, hid, heads=1, dropout=drop)
        elif kind == "sage":
            self.c1 = SAGEConv(d_in, hid); self.c2 = SAGEConv(hid, hid)
        else:
            self.c1 = GCNConv(d_in, hid); self.c2 = GCNConv(hid, hid)
        self.out = nn.Linear(hid, 1)
    def forward(self, x, ei):
        x = nn.functional.dropout(x, self.drop, self.training)
        h = nn.functional.elu(self.c1(x, ei))
        h = nn.functional.dropout(h, self.drop, self.training)
        h = nn.functional.elu(self.c2(h, ei))
        return self.out(h).squeeze(-1), h

def train_gnn2(kind, fit, seed, feats="std", epochs=None, lr=0.005, hid=64, drop=0.3,
               pw="sqrt", extra=None, init_c1=None, track=None):
    """Full-batch. Loss only on labelled `fit` nodes (+ optional weighted pseudo-labelled `extra`).
    Unknown / validation / test nodes take part in message passing through their features only."""
    epochs = epochs or GNN_EPOCHS
    torch.manual_seed(seed); np.random.seed(seed)
    x = GX[feats]
    model = GNN2(kind, x.shape[1], hid=hid, drop=drop).to(GDEV)
    if init_c1 is not None:
        model.c1.load_state_dict(init_c1)
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=5e-4)
    idx_np, y_np, w_np = np.asarray(fit), Y[fit].astype(np.float32), np.ones(len(fit), np.float32)
    if extra is not None:
        e_idx, e_y, e_w = extra
        idx_np = np.concatenate([idx_np, e_idx]); y_np = np.concatenate([y_np, e_y.astype(np.float32)])
        w_np = np.concatenate([w_np, e_w.astype(np.float32)])
    idx = torch.tensor(idx_np, device=GDEV); yt = torch.tensor(y_np, device=GDEV); wt = torch.tensor(w_np, device=GDEV)
    ratio = float((Y[fit] == 0).sum() / max((Y[fit] == 1).sum(), 1))
    pos_w = torch.tensor({"none": 1.0, "sqrt": ratio ** 0.5, "full": ratio}[pw], dtype=torch.float32, device=GDEV)
    curve = []
    for ep in range(1, epochs + 1):
        model.train(); opt.zero_grad()
        logit, _ = model(x, GEI)
        l = nn.functional.binary_cross_entropy_with_logits(logit[idx], yt, pos_weight=pos_w, reduction="none")
        ((l * wt).sum() / wt.sum()).backward(); opt.step()
        if track is not None and ep % 10 == 0:
            model.eval()
            with torch.no_grad():
                pv = model(x, GEI)[0][torch.tensor(track[0], device=GDEV)].cpu().numpy()
            curve.append((ep, roc_auc_score(track[1], pv)))
    model.eval()
    with torch.no_grad():
        logit, emb = model(x, GEI)
    return torch.sigmoid(logit).cpu().numpy().astype(np.float64), emb.cpu().numpy(), curve

def gnn2_fp(kind, **kw):
    def fp(tr, va, seed): return train_gnn2(kind, tr, seed, **kw)[0][va]
    return fp

def gnn2_stage1(kind, **kw):
    def f(fit, seed): return train_gnn2(kind, fit, seed, **kw)[0][:, None]
    return f

def run2(name, fp, seeds=(0,)):
    return evaluate(f"{name} [{len(seeds)}s]", fp, seeds=seeds)

torch device: cpu


In [46]:
for feats in ("std", "rank"):
    for s, k in (("roll", 1), ("gap", 1)):
        tr, va = fold_pos(DEV_FOLDS[s][k])
        _, _, cv = train_gnn2("gcn", tr, 0, feats=feats, epochs=150, track=(va, Y[va].astype(int)))
        print(feats, s, k, " ".join(f"{e}:{a:.3f}" for e, a in cv))

std roll 1 10:0.872 20:0.859 30:0.865 40:0.877 50:0.879 60:0.881 70:0.883 80:0.883 90:0.883 100:0.883 110:0.884 120:0.881 130:0.883 140:0.881 150:0.880
std gap 1 10:0.797 20:0.777 30:0.782 40:0.804 50:0.814 60:0.826 70:0.832 80:0.835 90:0.839 100:0.844 110:0.852 120:0.848 130:0.856 140:0.849 150:0.849
rank roll 1 10:0.795 20:0.810 30:0.818 40:0.827 50:0.831 60:0.834 70:0.835 80:0.835 90:0.835 100:0.835 110:0.835 120:0.833 130:0.833 140:0.833 150:0.832
rank gap 1 10:0.736 20:0.739 30:0.737 40:0.734 50:0.732 60:0.733 70:0.734 80:0.734 90:0.734 100:0.735 110:0.735 120:0.734 130:0.734 140:0.736 150:0.736


In [47]:
FEATS = "std"
GNN_EPOCHS = 120
for kind in ("gcn", "sage"):         
    run2(f"GNN {kind} ({FEATS})", gnn2_fp(kind, feats=FEATS))

GNN gcn (std) [1s]                               dev=0.9106 gap=0.8638 roll=0.9418 fold_std=0.0583 pr=0.7525 (503s) | d_gap=-0.0410 d_dev=-0.0383
GNN sage (std) [1s]                              dev=0.9362 gap=0.8949 roll=0.9637 fold_std=0.0519 pr=0.8355 (414s) | d_gap=-0.0099 d_dev=-0.0127


In [48]:
def ens_w(names, ws, sets=("roll", "gap")):
    out = {}
    for s in sets:
        vals = []
        for k, f in enumerate(DEV_FOLDS[s]):
            _, va = fold_pos(f); y = Y[va].astype(int)
            sc = sum(w * rankdata(PRED[n][(s, k)]) / len(va) for n, w in zip(names, ws))
            vals.append(roc_auc_score(y, sc))
        out[s] = float(np.mean(vals))
    out["dev"] = (3 * out["roll"] + 2 * out["gap"]) / 5
    return out

G_NAME = "GNN sage (std) [1s]"
for REF in ("LGBM | stack2 [3 seeds]", "LGBM | raw [3 seeds]"):
    print(REF)
    for w in (0, 0.1, 0.2, 0.3, 0.5):
        r = ens_w([REF, G_NAME], [1 - w, w])
        print(f"  w_sage={w:.1f}  roll={r['roll']:.4f}  gap={r['gap']:.4f}  dev={r['dev']:.4f}")

LGBM | stack2 [3 seeds]
  w_sage=0.0  roll=0.9762  gap=0.9053  dev=0.9479
  w_sage=0.1  roll=0.9764  gap=0.9060  dev=0.9483
  w_sage=0.2  roll=0.9762  gap=0.9062  dev=0.9482
  w_sage=0.3  roll=0.9757  gap=0.9060  dev=0.9478
  w_sage=0.5  roll=0.9742  gap=0.9049  dev=0.9465
LGBM | raw [3 seeds]
  w_sage=0.0  roll=0.9767  gap=0.8983  dev=0.9453
  w_sage=0.1  roll=0.9769  gap=0.8999  dev=0.9461
  w_sage=0.2  roll=0.9767  gap=0.9010  dev=0.9464
  w_sage=0.3  roll=0.9763  gap=0.9015  dev=0.9464
  w_sage=0.5  roll=0.9746  gap=0.9020  dev=0.9456


### Findings
* **Input scaling:** per-step rank features were much worse than standardised features for the GCN (roll fold 1: 0.835 vs 0.883, gap fold 1: 0.734 vs 0.85), as for the trees. The curves flatten after about 80 epochs on the roll fold and about 120 on the gap fold, so 120 epochs are used.
* **Models:** GCN scores dev 0.9106 / gap 0.8638 and GraphSAGE 0.9362 / 0.8949, against 0.9448 / 0.8979 for plain LightGBM. SAGE is much better than GCN because it keeps a node's own features separate from the neighbour average; GCN blurs them together.
* **Blend:** a rank blend of stack2 with SAGE gains at most +0.0009 gap and +0.0003 dev (weight 0.2). Blending SAGE into plain LightGBM gains +0.0037 gap at weight 0.5, but dev is unchanged. SAGE re-derives the neighbour information that stack2 already extracts, so it adds nothing.
* GAT, DGI pretraining, GNN self-training and a GNN-based first stage were not run, because the blend result showed that a GNN adds no new signal.

### 10.7 More graph features: extra 2-hop aggregates and neighbour aggregates of the top-X raw features

Two additions to stack2, each tested on its own and combined, with the same folds, seeds and acceptance rule as before.

1. **More 2-hop aggregates of the stage-1 score.** stack2 only has one 2-hop column (the mean of the neighbours' means). New columns, in three groups:
   * `strict`: mean, std and max of the score over nodes exactly two steps away (not the node itself, not a direct neighbour);
   * `directed`: mean over predecessors-of-predecessors and over successors-of-successors;
   * `shared`: mean over nodes that share a successor with the node, and over nodes that share a predecessor.
2. **Neighbour aggregates of the top-X raw features.** Rank the raw features by LightGBM gain importance, keep the top X, and add their mean over the node's neighbours (optionally max, predecessor/successor means, 2-hop mean, and "own value minus neighbour mean"). X is chosen from a grid by the gap folds; the smallest X within 0.002 of the best gap is taken, then confirmed on the three extra gap folds.

**Time-aware by construction.** The feature ranking is fitted only on the training rows of each fold. Edges only join transactions of the same time step, so neighbour aggregates never mix time steps. The aggregates use features and stage-1 scores of neighbours, never labels of validation or test rows. Neighbour blocks are blanked with the same row mask as the stack2 neighbour columns (train mask .25, validation mask `SIM_VAL_MASK`), so a no-extras run reproduces stack2 exactly.

In [14]:
SIM_VAL_MASK = 0.25
assert DECAY is None, "the new functions assume no recency weights (DECAY = None)"

# ---------------------------------------------------------------- extra adjacency matrices (all N x N, sparse)
def _binary(M):
    # 0/1 version of M without the diagonal (a node is not its own neighbour)
    M = M.tocsr().astype(np.float32)
    M.setdiag(0); M.eliminate_zeros(); M.data[:] = 1
    return M

A2_full = _binary(A_und @ A_und)                          # reachable in two undirected steps
A2_strict = (A2_full - A2_full.multiply(A_und)).tocsr()   # exactly two steps away: remove direct neighbours
A2_strict.eliminate_zeros(); A2_strict.data[:] = 1
A_pp = _binary(A_in @ A_in)     # predecessors of predecessors
A_ss = _binary(A_dir @ A_dir)   # successors of successors
A_cs = _binary(A_dir @ A_in)    # nodes that share at least one successor with i (i -> k <- j)
A_cp = _binary(A_in @ A_dir)    # nodes that share at least one predecessor with i (k -> i, k -> j)
print("average list size | 1-hop %.2f | strict 2-hop %.2f | pp %.2f | ss %.2f | share-succ %.2f | share-pred %.2f" %
      tuple(np.asarray(M.sum(1)).mean() for M in (A_und, A2_strict, A_pp, A_ss, A_cs, A_cp)))

# ---------------------------------------------------------------- aggregates of a node score over a neighbour list
def agg_block(M, p, stats=("mean",)):
    # For each row i of M, summarise p over the nodes listed in row i. Rows with an empty list get NaN.
    p = np.asarray(p, dtype=np.float64)
    d = np.asarray(M.sum(1)).ravel(); has = d > 0
    cols = []
    if "mean" in stats or "std" in stats:
        mean = np.full(N, np.nan); mean[has] = (M @ p)[has] / d[has]
    if "mean" in stats:
        cols.append(mean)
    if "std" in stats:
        sq = np.full(N, np.nan); sq[has] = (M @ (p * p))[has] / d[has]
        cols.append(np.sqrt(np.maximum(sq - mean ** 2, 0)))
    if "max" in stats:
        cols.append(row_max(M, p, fill=np.nan))
    return np.column_stack(cols)

TWO_HOP_GROUPS = {
    "strict":   [(A2_strict, ("mean", "std", "max"))],
    "directed": [(A_pp, ("mean",)), (A_ss, ("mean",))],
    "shared":   [(A_cp, ("mean",)), (A_cs, ("mean",))],
}
def two_hop_feats(p, groups):
    # Extra 2-hop columns of the stage-1 score p, for the chosen groups
    return np.hstack([agg_block(M, p, st) for g in groups for M, st in TWO_HOP_GROUPS[g]])

# ---------------------------------------------------------------- neighbour aggregates of the RAW features
_NB = {}
_NB_MAT = {"mean": A_und, "pred": A_in, "succ": A_dir, "two": A2_strict}
def nb_raw(kind):
    # Neighbour aggregate of EVERY raw feature (N x n_features). It uses no labels, so it is computed once and reused by all folds.
    # kind: "mean" (all neighbours), "pred", "succ", "two" (strict 2-hop mean) or "max" (max over all neighbours)
    if kind not in _NB:
        if kind == "max":
            out = np.column_stack([row_max(A_und, X_BASE[:, j].astype(np.float64), fill=np.nan) for j in range(X_BASE.shape[1])])
        else:
            M = _NB_MAT[kind]; d = np.asarray(M.sum(1)).ravel()
            out = (M @ X_BASE.astype(np.float32)) / np.maximum(d, 1)[:, None]
            out[d == 0] = np.nan                       # no neighbours: missing, as for the score aggregates
        _NB[kind] = out.astype(np.float32)
    return _NB[kind]

def raw_nbr_block(cols, kinds=("mean",), diff=False):
    # Columns `cols` of the raw features, aggregated over neighbours in each requested way
    parts = [nb_raw(k)[:, cols] for k in kinds]
    if diff:                                           # own value minus neighbour mean: trees cannot easily compute this themselves
        parts.append(X_BASE[:, cols] - nb_raw("mean")[:, cols])
    return np.hstack(parts)

_TOP = {}
def top_features(tr, seed, n_trees=100):
    # Raw-feature columns ranked by LightGBM gain importance, fitted on the TRAINING rows of the fold only (time-aware)
    key = (seed, int(T[tr].max()), len(tr))
    if key not in _TOP:
        m = lgb.LGBMClassifier(n_estimators=n_trees, learning_rate=0.1, num_leaves=31, subsample=0.8, subsample_freq=1,
                               colsample_bytree=0.8, importance_type="gain", random_state=seed, n_jobs=-1, verbose=-1)
        m.fit(X_BASE[tr], Y[tr].astype(int))
        _TOP[key] = np.argsort(-np.asarray(m.feature_importances_, dtype=float), kind="stable")
    return _TOP[key]

# ---------------------------------------------------------------- cache of the stage-1 out-of-fold scores
# Stage 1 does not depend on the stage-2 features being tested, so it is computed once per (fold, seed) and reused.
# Only stage-1 functions that carry a `.tag` are cached. Call _S1.clear() and _TOP.clear() if you change make_hgb or N_TREES.
_S1 = {}
def cached_oof(stage1, tr, seed, k):
    tag = getattr(stage1, "tag", None)
    if tag is None:
        return oof_scores(lambda fit: stage1(fit, seed), tr, k)
    key = (tag, seed, k, int(T[tr].max()), len(tr))
    if key not in _S1:
        _S1[key] = oof_scores(lambda fit: stage1(fit, seed), tr, k)
    return _S1[key]

S1_HGB = tree_stage1(make_hgb, X_BASE)
S1_HGB.tag = "hgb"

# ---------------------------------------------------------------- final-stage fit, with an optional (time-aware) resampler
def fit_predict(factory, X, tr, va, seed, sampler=None, n_bags=1):
    # Fit the final model on rows tr and score rows va. sampler(X, y, steps, w, seed) -> (X, y, steps, w) is applied to the
    # TRAINING rows only (section 10.8). With sampler=None and n_bags=1 this is identical to gbdt_fp.
    y = Y[tr].astype(int); out = np.zeros(len(va))
    for b in range(n_bags):
        sd = seed if n_bags == 1 else seed * 100 + b
        if sampler is None:
            Xf, yf, wf = X[tr], y, None
        else:
            Xf, yf, _, wf = sampler(X[tr], y, T[tr], None, sd)
        out += fit_model(factory, Xf, yf, wf, sd, False).predict_proba(X[va])[:, 1]
    return out / n_bags

def rs_gbdt_fp(factory, X, sampler, n_bags=1):
    # Plain model (no stacking) trained on resampled training rows
    return lambda tr, va, seed: fit_predict(factory, X, tr, va, seed, sampler, n_bags)

# ---------------------------------------------------------------- stack4 = stack2 + optional extras
def stacked_fp4(stage1, X_base, final_factory, k=4, p_mask=0.25,
                two_hop=(), n_top=0, nb_kinds=("mean",), diff=False, sampler=None, n_bags=1):
    # two_hop : groups of extra 2-hop score aggregates, any of "strict", "directed", "shared"
    # n_top   : number of top raw features whose neighbour aggregates are added (0 = none)
    # nb_kinds: which neighbour aggregates of those features: "mean", "max", "pred", "succ", "two"
    # diff    : also add (own value - neighbour mean) for those features
    # With every option off, the features and the random masks are exactly those of stacked_fp2.
    assert X_base is X_BASE, "the raw-feature neighbour aggregates are built from X_BASE"
    def fp(tr, va, seed):
        S = cached_oof(stage1, tr, seed, k)                       # out-of-fold stage-1 score
        agg = nbr_aggregate(S[:, 0])
        agg[und_deg == 0] = np.nan                                # real isolates: missing, not "-1"
        blocks = [agg]
        if two_hop:
            blocks.append(two_hop_feats(S[:, 0], two_hop))
        if n_top > 0:
            blocks.append(raw_nbr_block(top_features(tr, seed)[:n_top], nb_kinds, diff).astype(np.float64))
        rng = np.random.RandomState(seed)                         # same random draws as stacked_fp2
        m = np.zeros(N, bool)
        m[tr] = rng.rand(len(tr)) < p_mask                        # train mask
        m[va] = rng.rand(len(va)) < SIM_VAL_MASK                  # validation mask
        for b in blocks:
            b[m] = np.nan
        X = np.hstack([X_base, S] + blocks).astype(np.float32)
        return fit_predict(final_factory, X, tr, va, seed, sampler, n_bags)
    return fp

# ---------------------------------------------------------------- reporting helpers
def run4(tag, stage1=S1_HGB, **kw):
    return evaluate(f"LGBM | stack4 {tag} [3 seeds]", stacked_fp4(stage1, X_BASE, MAIN_FACTORY, **kw), seeds=(0, 1, 2))

def summarize(ref, prefix):
    # Table of every evaluated model whose name contains `prefix`, with the change in dev and gap versus `ref`
    L = leaderboard(); r = L[L.model == ref].iloc[0]
    d = L[L.model.str.contains(prefix, regex=False)][["model", "roc_dev", "roc_gap", "fold_std"]].copy()
    d["d_dev"] = d.roc_dev - r.roc_dev; d["d_gap"] = d.roc_gap - r.roc_gap
    print(f"reference {ref}: dev={r.roc_dev:.4f} gap={r.roc_gap:.4f}")
    print(d.round(4).to_string(index=False))

def verdict(cand, ref):
    # The acceptance rule used in this notebook: gap gain >= 0.005, dev not lower, and wins >= 7 of the 8 folds.
    # Both models must already have been run through confirm() / evalx() so that the 3 extra gap folds exist.
    L = leaderboard().set_index("model")
    d_gap = L.loc[cand, "roc_gap"] - L.loc[ref, "roc_gap"]; d_dev = L.loc[cand, "roc_dev"] - L.loc[ref, "roc_dev"]
    wins = int((fold_aucs(cand) > fold_aucs(ref)).sum())
    ok = (d_gap >= 0.005) and (d_dev >= 0) and (wins >= 7)
    print(f"d_gap={d_gap:+.4f} d_dev={d_dev:+.4f} wins {wins}/8 -> {'ACCEPT' if ok else 'REJECT (does not meet the rule)'}")

average list size | 1-hop 2.24 | strict 2-hop 17.69 | pp 1.10 | ss 1.10 | share-succ 12.22 | share-pred 3.63


In [15]:
# Sanity check: with every option off, stack4 must reproduce stack2 (same features, same masks, same seeds)
SIM_VAL_MASK = 0.25
REF = "LGBM | stack2 [3 seeds]"
run4("no extras (must equal stack2)")
summarize(REF, "stack4 no extras")

LGBM | stack4 no extras (must equal stack2) [3 seeds] dev=0.9478 gap=0.9059 roll=0.9757 fold_std=0.0508 pr=0.9047 (153s)
reference LGBM | stack2 [3 seeds]: dev=0.9478 gap=0.9059
                                                model  roc_dev  roc_gap  fold_std  d_dev  d_gap
LGBM | stack4 no extras (must equal stack2) [3 seeds]   0.9478   0.9059    0.0508    0.0    0.0


In [16]:
# A) Extra 2-hop aggregates of the stage-1 score, one group at a time and together
SIM_VAL_MASK = 0.25
run4("2hop strict",   two_hop=("strict",))
run4("2hop directed", two_hop=("directed",))
run4("2hop shared",   two_hop=("shared",))
run4("2hop all",      two_hop=("strict", "directed", "shared"))
summarize(REF, "stack4 2hop")

LGBM | stack4 2hop strict [3 seeds]              dev=0.9501 gap=0.9113 roll=0.9760 fold_std=0.0477 pr=0.9060 (36s)
LGBM | stack4 2hop directed [3 seeds]            dev=0.9525 gap=0.9171 roll=0.9761 fold_std=0.0422 pr=0.9074 (49s)
LGBM | stack4 2hop shared [3 seeds]              dev=0.9531 gap=0.9173 roll=0.9769 fold_std=0.0423 pr=0.9081 (70s)
LGBM | stack4 2hop all [3 seeds]                 dev=0.9512 gap=0.9103 roll=0.9784 fold_std=0.0459 pr=0.9071 (115s)
reference LGBM | stack2 [3 seeds]: dev=0.9478 gap=0.9059
                                model  roc_dev  roc_gap  fold_std  d_dev  d_gap
  LGBM | stack4 2hop shared [3 seeds]   0.9531   0.9173    0.0423 0.0053 0.0114
LGBM | stack4 2hop directed [3 seeds]   0.9525   0.9171    0.0422 0.0047 0.0112
  LGBM | stack4 2hop strict [3 seeds]   0.9501   0.9113    0.0477 0.0023 0.0053
     LGBM | stack4 2hop all [3 seeds]   0.9512   0.9103    0.0459 0.0034 0.0044


In [18]:
# B) Neighbour mean of the top-X raw features: choose X from a grid
SIM_VAL_MASK = 0.25
TOP_GRID = (5, 10, 20, 40, 80)
for n in TOP_GRID:
    run4(f"top{n} mean", n_top=n)
summarize(REF, "stack4 top")

def pick_n_top(grid, suffix=" mean [3 seeds]", tol=0.002):
    # Smallest X whose gap score is within `tol` of the best gap score in the grid (fewer features = less overfitting)
    L = leaderboard().set_index("model")
    gaps = {n: float(L.loc[f"LGBM | stack4 top{n}{suffix}", "roc_gap"]) for n in grid}
    best = max(gaps.values())
    chosen = min(n for n, g in gaps.items() if g >= best - tol)
    print("gap by X:", {n: round(g, 4) for n, g in gaps.items()}, "| best %.4f | chosen X = %d" % (best, chosen))
    return chosen

N_TOP = pick_n_top(TOP_GRID)

LGBM | stack4 top5 mean [3 seeds]                dev=0.9507 gap=0.9120 roll=0.9765 fold_std=0.0467 pr=0.9059 (50s)
LGBM | stack4 top10 mean [3 seeds]               dev=0.9469 gap=0.9044 roll=0.9752 fold_std=0.0503 pr=0.9045 (80s)
LGBM | stack4 top20 mean [3 seeds]               dev=0.9498 gap=0.9109 roll=0.9757 fold_std=0.0465 pr=0.9059 (84s)
LGBM | stack4 top40 mean [3 seeds]               dev=0.9476 gap=0.9021 roll=0.9779 fold_std=0.0513 pr=0.9047 (90s)
LGBM | stack4 top80 mean [3 seeds]               dev=0.9450 gap=0.9015 roll=0.9740 fold_std=0.0535 pr=0.9039 (99s)
reference LGBM | stack2 [3 seeds]: dev=0.9478 gap=0.9059
                             model  roc_dev  roc_gap  fold_std   d_dev   d_gap
 LGBM | stack4 top5 mean [3 seeds]   0.9507   0.9120    0.0467  0.0029  0.0061
LGBM | stack4 top20 mean [3 seeds]   0.9498   0.9109    0.0465  0.0020  0.0050
LGBM | stack4 top10 mean [3 seeds]   0.9469   0.9044    0.0503 -0.0009 -0.0015
LGBM | stack4 top40 mean [3 seeds]   0.9476   0.9021

In [19]:
# C) Variants of the neighbour features of the top-N_TOP raw features, and everything together
SIM_VAL_MASK = 0.25
run4(f"top{N_TOP} mean+max",    n_top=N_TOP, nb_kinds=("mean", "max"))
run4(f"top{N_TOP} pred+succ",   n_top=N_TOP, nb_kinds=("pred", "succ"))
run4(f"top{N_TOP} mean+2hop",   n_top=N_TOP, nb_kinds=("mean", "two"))
run4(f"top{N_TOP} mean+diff",   n_top=N_TOP, nb_kinds=("mean",), diff=True)
run4(f"top{N_TOP} mean + 2hop all", n_top=N_TOP, two_hop=("strict", "directed", "shared"))
summarize(REF, f"stack4 top{N_TOP}")

LGBM | stack4 top5 mean+max [3 seeds]            dev=0.9490 gap=0.9085 roll=0.9760 fold_std=0.0481 pr=0.9049 (32s)
LGBM | stack4 top5 pred+succ [3 seeds]           dev=0.9511 gap=0.9110 roll=0.9778 fold_std=0.0462 pr=0.9062 (82s)
LGBM | stack4 top5 mean+2hop [3 seeds]           dev=0.9472 gap=0.9053 roll=0.9751 fold_std=0.0504 pr=0.9044 (35s)
LGBM | stack4 top5 mean+diff [3 seeds]           dev=0.9505 gap=0.9087 roll=0.9784 fold_std=0.0485 pr=0.9056 (48s)
LGBM | stack4 top5 mean + 2hop all [3 seeds]     dev=0.9522 gap=0.9155 roll=0.9767 fold_std=0.0452 pr=0.9071 (79s)
reference LGBM | stack2 [3 seeds]: dev=0.9478 gap=0.9059
                                       model  roc_dev  roc_gap  fold_std   d_dev   d_gap
LGBM | stack4 top5 mean + 2hop all [3 seeds]   0.9522   0.9155    0.0452  0.0044  0.0096
           LGBM | stack4 top5 mean [3 seeds]   0.9507   0.9120    0.0467  0.0029  0.0061
      LGBM | stack4 top5 pred+succ [3 seeds]   0.9511   0.9110    0.0462  0.0033  0.0051
      LGBM |

In [20]:
# D) Confirm the best candidate on the three extra gap folds (8 folds in total).
# Replace CAND_TAG by the variant you want to confirm (the part of the name between "stack4 " and " [3 seeds]").
SIM_VAL_MASK = 0.25
CAND_TAG, CAND_KW = f"top{N_TOP} mean", dict(n_top=N_TOP)          # e.g. f"top{N_TOP} mean+diff", dict(n_top=N_TOP, diff=True)
CAND = f"LGBM | stack4 {CAND_TAG} [3 seeds]"
confirm(CAND, stacked_fp4(S1_HGB, X_BASE, MAIN_FACTORY, **CAND_KW),
        REF, stacked_fp2(tree_stage1(make_hgb, X_BASE), X_BASE, MAIN_FACTORY, decay=DECAY, p_mask=0.25))
verdict(CAND, REF)

LGBM | stack2 [3 seeds] [gapx]                   dev=0.9097 gap=nan roll=nan fold_std=0.0864 pr=0.8483 (134s)
LGBM | stack4 top5 mean [3 seeds] [gapx]         dev=0.9026 gap=nan roll=nan fold_std=0.0914 pr=0.8456 (88s)
             a       b    diff
roll 0  0.9950  0.9950 -0.0001
     1  0.9372  0.9364  0.0008
     2  0.9973  0.9972  0.0001
gap  0  0.9415  0.9396  0.0018
     1  0.8881  0.8710  0.0170
gapx 0  0.9911  0.9916 -0.0005
     1  0.9156  0.9255 -0.0099
     2  0.8089  0.8172 -0.0083
'LGBM | stack4 top5 mean [3 seeds]' wins 4 of 8 folds | mean diff +0.0001
d_gap=+0.0061 d_dev=+0.0029 wins 4/8 -> REJECT (does not meet the rule)


In [21]:
SIM_VAL_MASK = 0.25

# 1) free check: fold-by-fold differences on the 5 dev folds (no new fitting)
def dev_diff(a, b):
    rows = []
    for s in ("roll", "gap"):
        for k, f in enumerate(DEV_FOLDS[s]):
            _, va = fold_pos(f); y = Y[va].astype(int)
            rows.append(dict(set=s, fold=k, a=roc_auc_score(y, PRED[a][(s, k)]), b=roc_auc_score(y, PRED[b][(s, k)])))
    d = pd.DataFrame(rows); d["diff"] = d.a - d.b
    print(d.round(4)); print(f"wins {(d['diff'] > 0).sum()} of {len(d)} dev folds")

for g in ("shared", "directed"):
    print("\n==", g)
    dev_diff(f"LGBM | stack4 2hop {g} [3 seeds]", REF)

# 2) confirmation on the 3 extra gap folds
for g in ("shared", "directed"):
    name = f"LGBM | stack4 2hop {g} [3 seeds]"
    evalx(name, stacked_fp4(S1_HGB, X_BASE, MAIN_FACTORY, two_hop=(g,)))
    compare8(name, REF)
    verdict(name, REF)


== shared
    set  fold       a       b    diff
0  roll     0  0.9950  0.9950  0.0000
1  roll     1  0.9409  0.9364  0.0045
2  roll     2  0.9978  0.9972  0.0006
3   gap     0  0.9381  0.9396 -0.0015
4   gap     1  0.8962  0.8710  0.0252
wins 4 of 5 dev folds

== directed
    set  fold       a       b    diff
0  roll     0  0.9945  0.9950 -0.0005
1  roll     1  0.9403  0.9364  0.0039
2  roll     2  0.9975  0.9972  0.0003
3   gap     0  0.9355  0.9396 -0.0041
4   gap     1  0.9012  0.8710  0.0301
wins 3 of 5 dev folds
LGBM | stack4 2hop shared [3 seeds] [gapx]       dev=0.9078 gap=nan roll=nan fold_std=0.0903 pr=0.8469 (20s)
             a       b    diff
roll 0  0.9950  0.9950  0.0000
     1  0.9409  0.9364  0.0045
     2  0.9978  0.9972  0.0006
gap  0  0.9381  0.9396 -0.0015
     1  0.8962  0.8710  0.0252
gapx 0  0.9914  0.9916 -0.0002
     1  0.9259  0.9255  0.0004
     2  0.8101  0.8172 -0.0071
'LGBM | stack4 2hop shared [3 seeds]' wins 5 of 8 folds | mean diff +0.0027
d_gap=+0.011

### Findings
Reference: stack2 over 3 seeds, dev 0.9478 / gap 0.9059.

| Variant | Dev | Gap | Verdict |
|---|---|---|---|
| 2-hop strict | 0.9501 | 0.9113 | below the bar |
| 2-hop directed | 0.9525 | 0.9171 | fails 8-fold check (wins 4 of 8) |
| 2-hop shared | 0.9531 | 0.9173 | fails 8-fold check (wins 5 of 8) |
| 2-hop all three | 0.9512 | 0.9103 | worse than its parts |
| top-X neighbour mean, X = 5 / 10 / 20 / 40 / 80 | 0.9507 / 0.9469 / 0.9498 / 0.9476 / 0.9450 | 0.9120 / 0.9044 / 0.9109 / 0.9021 / 0.9015 | no trend with X; X = 5 fails the 8-fold check (wins 4 of 8) |

* **The gains come from one fold.** Gap fold 1 improves by +0.025 (shared) and +0.030 (directed); over the other seven folds the mean difference is about zero (-0.0005 and -0.0001).
* **No trend with X.** The gap score goes up and down as X grows, so the best X is the luckiest of five tries, not a signal.
* **Rejected.** Stack2 is unchanged.

### 10.8 Time-aware undersampling and oversampling

Re-sampling is made time-aware here:

1. Applied to training rows only, one step at a time. Validation and test rows are never touched
2. recency variable: with `recency = r < 1`, a step that is `a` steps older than the last training step gets its sampling effort scaled by `r**a` (older steps lose more negatives and gain fewer copies/synthetic rows).
3. When a stage-1 model is resampled (inside `oof_scores`), the sampler only sees the rows that model is fitted on, so out-of-fold scores stay out-of-fold.

Methods: random **undersampling** of licit rows (`under`: per-step balance of `neg_per_pos` licit rows per illicit row, or `uniform` keep-rate that preserves each step's share), random **oversampling** (`over`: duplicate illicit rows), per-step **SMOTE** (`smote`), and chains such as undersample then SMOTE.

In [22]:
from sklearn.neighbors import NearestNeighbors

# A sampler is a function sampler(X, y, steps, w, seed) -> (X, y, steps, w).
#   X, y, steps, w : feature rows, labels, time step and weight of the TRAINING rows (w may be None = all ones)
# Rows are only added or removed; new rows get the time step of the rows they were built from.

def _w(w, n):
    return np.ones(n) if w is None else np.asarray(w, dtype=float).copy()

def _mult(t, tmax, mult, recency):
    # how strongly step t is oversampled: 1 + (mult-1) * recency**age  (recency=None: the same for every step)
    return 1.0 + (mult - 1.0) * (1.0 if recency is None else recency ** (tmax - t))

def under_sampler(neg_per_pos=3.0, mode="per_step", recency=None, min_keep=0.05, reweight=False):
    # Randomly drop licit rows, separately inside each time step.
    #   mode="per_step": keep neg_per_pos licit rows per illicit row of that step (balances every step)
    #   mode="uniform" : keep the same fraction of licit rows in every step (keeps each step's share of the data)
    #   recency        : older steps keep fewer licit rows (kept count x recency**age)
    #   min_keep       : never keep fewer than this fraction of a step's licit rows (steps with few illicit rows)
    #   reweight       : give kept licit rows weight 1/keep-rate, which restores the original class prior
    def s(X, y, steps, w, seed):
        w = _w(w, len(y)); rng = np.random.RandomState(seed + 7); tmax = steps.max()
        rate = min(1.0, neg_per_pos * (y == 1).sum() / max((y == 0).sum(), 1))
        keep = np.zeros(len(y), bool)
        for t in np.unique(steps):
            idx = np.where(steps == t)[0]
            pos, neg = idx[y[idx] == 1], idx[y[idx] == 0]
            keep[pos] = True
            n_keep = neg_per_pos * len(pos) if mode == "per_step" else rate * len(neg)
            if recency is not None:
                n_keep *= recency ** (tmax - t)
            n_keep = int(round(min(len(neg), max(n_keep, min_keep * len(neg)))))
            chosen = rng.choice(neg, n_keep, replace=False) if n_keep < len(neg) else neg
            keep[chosen] = True
            if reweight and n_keep > 0:
                w[chosen] *= len(neg) / n_keep
        return X[keep], y[keep], steps[keep], w[keep]
    return s

def over_sampler(mult=3.0, recency=None):
    # Randomly duplicate illicit rows, separately inside each time step, until each step has about `mult` x its illicit rows
    def s(X, y, steps, w, seed):
        w = _w(w, len(y)); rng = np.random.RandomState(seed + 11); tmax = steps.max(); extra = []
        for t in np.unique(steps):
            pos = np.where((steps == t) & (y == 1))[0]
            n_extra = int(round((_mult(t, tmax, mult, recency) - 1.0) * len(pos)))
            if len(pos) and n_extra > 0:
                extra.append(rng.choice(pos, n_extra, replace=True))
        if not extra:
            return X, y, steps, w
        e = np.concatenate(extra)
        return np.vstack([X, X[e]]), np.concatenate([y, y[e]]), np.concatenate([steps, steps[e]]), np.concatenate([w, w[e]])
    return s

def smote_sampler(mult=2.0, k=5, knn_cols=None, recency=None):
    # SMOTE inside each time step: a new illicit row lies on the line between an illicit row and one of its k nearest
    # illicit neighbours FROM THE SAME STEP. Nearest neighbours use the first `knn_cols` columns (the raw features; they have
    # no NaN), standardised. Every column of the new row is interpolated; a column is NaN if either parent is NaN.
    def s(X, y, steps, w, seed):
        w = _w(w, len(y)); rng = np.random.RandomState(seed + 13); tmax = steps.max()
        d0 = X.shape[1] if knn_cols is None else knn_cols
        mu = X[:, :d0].mean(0); sd = X[:, :d0].std(0) + 1e-6
        newX, newS = [], []
        for t in np.unique(steps):
            pos = np.where((steps == t) & (y == 1))[0]
            n_new = int(round((_mult(t, tmax, mult, recency) - 1.0) * len(pos)))
            if len(pos) < 2 or n_new <= 0:
                continue                                             # SMOTE needs at least two illicit rows in the step
            Z = (X[pos, :d0] - mu) / sd
            nbr = NearestNeighbors(n_neighbors=min(k + 1, len(pos))).fit(Z).kneighbors(Z, return_distance=False)[:, 1:]
            base = rng.randint(0, len(pos), n_new)
            pick = nbr[base, rng.randint(0, nbr.shape[1], n_new)]
            lam = rng.rand(n_new, 1)
            A, B = X[pos[base]].astype(np.float64), X[pos[pick]].astype(np.float64)
            newX.append((A + lam * (B - A)).astype(X.dtype)); newS.append(np.full(n_new, t))
        if not newX:
            return X, y, steps, w
        nx_ = np.vstack(newX); ns_ = np.concatenate(newS)
        return (np.vstack([X, nx_]), np.concatenate([y, np.ones(len(nx_), dtype=y.dtype)]),
                np.concatenate([steps, ns_]), np.concatenate([w, np.ones(len(nx_))]))
    return s

def chain(*samplers):
    # Apply several samplers one after the other (e.g. undersample, then SMOTE)
    def s(X, y, steps, w, seed):
        for i, f in enumerate(samplers):
            X, y, steps, w = f(X, y, steps, w, seed + i)
        return X, y, steps, w
    return s

def tree_stage1_rs(factory, X, sampler, tag):
    # Stage-1 model fitted on resampled rows. Pass it to stacked_fp4 as stage1; `tag` keeps its cached scores separate.
    def f(fit, seed):
        Xf, yf, _, wf = sampler(X[fit], Y[fit].astype(int), T[fit], None, seed)
        return fit_model(factory, Xf, yf, wf, seed, False).predict_proba(X)[:, 1:2]
    f.tag = tag
    return f

D0 = X_BASE.shape[1]
SAMPLERS = {
    "under 1:1":                 under_sampler(1.0),
    "under 3:1":                 under_sampler(3.0),
    "under 10:1":                under_sampler(10.0),
    "under 3:1 uniform":         under_sampler(3.0, mode="uniform"),
    "under 3:1 recency .9":      under_sampler(3.0, recency=0.9),
    "under 3:1 reweighted":      under_sampler(3.0, reweight=True),
    "over x2":                   over_sampler(2.0),
    "over x4":                   over_sampler(4.0),
    "over x4 recency .9":        over_sampler(4.0, recency=0.9),
    "smote x2":                  smote_sampler(2.0, knn_cols=D0),
    "smote x4":                  smote_sampler(4.0, knn_cols=D0),
    "smote x4 recency .9":       smote_sampler(4.0, knn_cols=D0, recency=0.9),
    "under 3:1 + smote x2":      chain(under_sampler(3.0), smote_sampler(2.0, knn_cols=D0)),
}

In [23]:
# Check that the samplers behave as described, on one fold (gap fold 0): counts per time step before / after, and
# that synthetic SMOTE rows lie inside the range of the illicit rows of their own step.
tr_c, va_c = fold_pos(DEV_FOLDS["gap"][0])
assert T[tr_c].max() < T[va_c].min(), "training steps must all come before the validation steps"
y_tr = Y[tr_c].astype(int)
for nm in ("under 3:1", "over x4", "smote x4"):
    Xr, yr, sr, wr = SAMPLERS[nm](X_BASE[tr_c], y_tr, T[tr_c], None, 0)
    assert sr.min() >= T[tr_c].min() and sr.max() <= T[tr_c].max(), "a sampler produced a row outside the training steps"
    tab = pd.DataFrame({"pos_before": pd.Series(y_tr).groupby(T[tr_c]).sum(), "neg_before": pd.Series(1 - y_tr).groupby(T[tr_c]).sum(),
                        "pos_after": pd.Series(yr).groupby(sr).sum(), "neg_after": pd.Series(1 - yr).groupby(sr).sum()}).fillna(0).astype(int)
    print(f"--- {nm}: rows {len(y_tr):,} -> {len(yr):,}, illicit share {y_tr.mean():.3f} -> {yr.mean():.3f}")
    print(tab.T.to_string())
    if nm.startswith("smote"):
        syn = slice(len(tr_c), None)                                   # synthetic rows are appended after the original rows
        for t_ in np.unique(sr[syn]):
            pos_rows = X_BASE[tr_c][(T[tr_c] == t_) & (y_tr == 1)]; syn_rows = Xr[syn][sr[syn] == t_]
            tol = 1e-4 * (np.abs(pos_rows).max(0) + 1)
            assert np.all(syn_rows >= pos_rows.min(0) - tol) and np.all(syn_rows <= pos_rows.max(0) + tol), f"synthetic row outside the illicit rows of step {t_}"
        print("SMOTE check passed: every synthetic row lies inside the range of the illicit rows of its own step")

--- under 3:1: rows 13,670 -> 2,635, illicit share 0.049 -> 0.255
              1     2     3     4     5    6     7     8    9    10   11   12
pos_before    17    18    11    30     8    5   102    67  248   18  131   16
neg_before  2130  1099  1268  1410  1874  480  1101  1098  530  954  565  490
pos_after     17    18    11    30     8    5   102    67  248   18  131   16
neg_after    106    55    63    90    94   24   306   201  530   54  393   48
--- over x4: rows 13,670 -> 15,683, illicit share 0.049 -> 0.171
              1     2     3     4     5    6     7     8    9    10   11   12
pos_before    17    18    11    30     8    5   102    67  248   18  131   16
neg_before  2130  1099  1268  1410  1874  480  1101  1098  530  954  565  490
pos_after     68    72    44   120    32   20   408   268  992   72  524   64
neg_after   2130  1099  1268  1410  1874  480  1101  1098  530  954  565  490
--- smote x4: rows 13,670 -> 15,683, illicit share 0.049 -> 0.171
              1     2  

In [24]:
# A) Resampling on its own (plain LightGBM on the raw features), three seeds each
SIM_VAL_MASK = 0.25
RAW_REF = "LGBM | raw [3 seeds]"
for nm, smp in SAMPLERS.items():
    evaluate(f"LGBM | raw + rs {nm} [3 seeds]", rs_gbdt_fp(MAIN_FACTORY, X_BASE, smp), seeds=(0, 1, 2))

# Time-aware bagging: 8 models, each trained on a different per-step undersample, scores averaged
evaluate("LGBM | raw + rs under 3:1 x8 bags [3 seeds]", rs_gbdt_fp(MAIN_FACTORY, X_BASE, SAMPLERS["under 3:1"], n_bags=8), seeds=(0, 1, 2))
summarize(RAW_REF, "LGBM | raw + rs ")

LGBM | raw + rs under 1:1 [3 seeds]              dev=0.9440 gap=0.8951 roll=0.9766 fold_std=0.0563 pr=0.9007 (17s)
LGBM | raw + rs under 3:1 [3 seeds]              dev=0.9425 gap=0.8934 roll=0.9752 fold_std=0.0569 pr=0.8996 (19s)
LGBM | raw + rs under 10:1 [3 seeds]             dev=0.9463 gap=0.8985 roll=0.9782 fold_std=0.0534 pr=0.9042 (21s)
LGBM | raw + rs under 3:1 uniform [3 seeds]      dev=0.9452 gap=0.8967 roll=0.9776 fold_std=0.0554 pr=0.9006 (20s)
LGBM | raw + rs under 3:1 recency .9 [3 seeds]   dev=0.9396 gap=0.8855 roll=0.9757 fold_std=0.0603 pr=0.8984 (18s)
LGBM | raw + rs under 3:1 reweighted [3 seeds]   dev=0.9464 gap=0.8982 roll=0.9785 fold_std=0.0543 pr=0.9018 (20s)
LGBM | raw + rs over x2 [3 seeds]                dev=0.9423 gap=0.8902 roll=0.9770 fold_std=0.0608 pr=0.9012 (65s)
LGBM | raw + rs over x4 [3 seeds]                dev=0.9411 gap=0.8889 roll=0.9758 fold_std=0.0599 pr=0.8968 (74s)
LGBM | raw + rs over x4 recency .9 [3 seeds]     dev=0.9381 gap=0.8790 roll=0.97

In [ ]:
# B) Resampling inside the stacked model: (i) final stage only, (ii) stage 1 only
SIM_VAL_MASK = 0.25
S2_SAMPLERS = ["under 3:1", "under 3:1 recency .9", "over x2", "smote x2", "smote x4", "under 3:1 + smote x2"]
for nm in S2_SAMPLERS:
    run4(f"rs final {nm}", sampler=SAMPLERS[nm])
for nm in ("under 3:1", "smote x2"):
    s1 = tree_stage1_rs(make_hgb, X_BASE, SAMPLERS[nm], tag=f"hgb_rs_{nm}")
    run4(f"rs stage1 {nm}", stage1=s1)
summarize(REF, "stack4 rs ")

LGBM | stack4 rs final under 3:1 [3 seeds]       dev=0.9478 gap=0.9057 roll=0.9758 fold_std=0.0477 pr=0.9045 (58s)
LGBM | stack4 rs final under 3:1 recency .9 [3 seeds] dev=0.9442 gap=0.8972 roll=0.9756 fold_std=0.0516 pr=0.9027 (49s)
LGBM | stack4 rs final over x2 [3 seeds]         dev=0.9499 gap=0.9081 roll=0.9778 fold_std=0.0467 pr=0.9060 (71s)
LGBM | stack4 rs final smote x2 [3 seeds]        dev=0.9504 gap=0.9135 roll=0.9749 fold_std=0.0450 pr=0.9060 (60s)
LGBM | stack4 rs final under 3:1 + smote x2 [3 seeds] dev=0.9491 gap=0.9077 roll=0.9768 fold_std=0.0486 pr=0.9044 (32s)
LGBM | stack4 rs stage1 under 3:1 [3 seeds]      dev=0.9468 gap=0.9022 roll=0.9765 fold_std=0.0502 pr=0.9043 (216s)
LGBM | stack4 rs stage1 smote x2 [3 seeds]       dev=0.9470 gap=0.9009 roll=0.9778 fold_std=0.0533 pr=0.9047 (235s)
reference LGBM | stack2 [3 seeds]: dev=0.9478 gap=0.9059
                                                model  roc_dev  roc_gap  fold_std   d_dev   d_gap
            LGBM | stack4 rs

In [ ]:
SIM_VAL_MASK = 0.25

# free check: where does the SMOTE x4 gain come from?
dev_diff("LGBM | raw + rs smote x4 [3 seeds]", RAW_REF)

def cross_step(sampler):
    # same oversampler, but all time steps are treated as ONE group, so synthetic rows may mix steps
    # (only for over/SMOTE samplers, which append rows)
    def s(X, y, steps, w, seed):
        X2, y2, _, w2 = sampler(X, y, np.zeros_like(steps), w, seed)
        return X2, y2, np.concatenate([steps, np.full(len(y2) - len(y), steps.max())]), w2
    return s

SAMPLERS.update({
    "smote x6":            smote_sampler(6.0,  knn_cols=D0),
    "smote x8":            smote_sampler(8.0,  knn_cols=D0),
    "smote x12":           smote_sampler(12.0, knn_cols=D0),
    "smote x4 cross-step": cross_step(smote_sampler(4.0, knn_cols=D0)),
    "smote x8 cross-step": cross_step(smote_sampler(8.0, knn_cols=D0)),
})
for nm in ("smote x6", "smote x8", "smote x12", "smote x4 cross-step", "smote x8 cross-step"):
    evaluate(f"LGBM | raw + rs {nm} [3 seeds]", rs_gbdt_fp(MAIN_FACTORY, X_BASE, SAMPLERS[nm]), seeds=(0, 1, 2))
summarize(RAW_REF, "LGBM | raw + rs smote")

LGBM | raw + rs smote x4 [3 seeds] [gapx]        dev=0.9168 gap=nan roll=nan fold_std=0.0827 pr=0.8515 (23s)
             a       b    diff
roll 0  0.9940  0.9940 -0.0000
     1  0.9415  0.9386  0.0028
     2  0.9976  0.9974  0.0002
gap  0  0.9475  0.9407  0.0069
     1  0.8883  0.8560  0.0323
gapx 0  0.9899  0.9917 -0.0017
     1  0.9348  0.9266  0.0082
     2  0.8271  0.8257  0.0014
'LGBM | raw + rs smote x4 [3 seeds]' wins 6 of 8 folds | mean diff +0.0063
d_gap=+0.0204 d_dev=+0.0090 wins 6/8 -> REJECT (does not meet the rule)
LGBM | stack4 rs final smote x4 [3 seeds] [gapx] dev=0.8998 gap=nan roll=nan fold_std=0.0992 pr=0.8433 (23s)
             a       b    diff
roll 0  0.9948  0.9950 -0.0002
     1  0.9385  0.9364  0.0021
     2  0.9971  0.9972 -0.0002
gap  0  0.9393  0.9396 -0.0003
     1  0.8824  0.8710  0.0114
gapx 0  0.9906  0.9916 -0.0010
     1  0.9208  0.9255 -0.0047
     2  0.7936  0.8172 -0.0236
'LGBM | stack4 rs final smote x4 [3 seeds]' wins 2 of 8 folds | mean diff -0.

In [ ]:
# C) Confirm the best resampling variant on the three extra gap folds (8 folds in total).
# Choose ONE of the two blocks: plain model against LGBM | raw, or stacked model against stack2.
SIM_VAL_MASK = 0.25
BEST_RS = "smote x4"                                             # replace by the winner from A) or B)

# plain model
CAND_RS = f"LGBM | raw + rs {BEST_RS} [3 seeds]"
confirm(CAND_RS, rs_gbdt_fp(MAIN_FACTORY, X_BASE, SAMPLERS[BEST_RS]), RAW_REF, gbdt_fp(MAIN_FACTORY, X_BASE, decay=DECAY))
verdict(CAND_RS, RAW_REF)

# stacked model (final-stage resampling)
CAND_RS2 = f"LGBM | stack4 rs final {BEST_RS} [3 seeds]"
confirm(CAND_RS2, stacked_fp4(S1_HGB, X_BASE, MAIN_FACTORY, sampler=SAMPLERS[BEST_RS]),
        REF, stacked_fp2(tree_stage1(make_hgb, X_BASE), X_BASE, MAIN_FACTORY, decay=DECAY, p_mask=0.25))
verdict(CAND_RS2, REF)

    set  fold       a       b    diff
0  roll     0  0.9940  0.9940 -0.0000
1  roll     1  0.9415  0.9386  0.0028
2  roll     2  0.9976  0.9974  0.0002
3   gap     0  0.9475  0.9407  0.0069
4   gap     1  0.8883  0.8560  0.0323
wins 4 of 5 dev folds
LGBM | raw + rs smote x6 [3 seeds]               dev=0.9530 gap=0.9132 roll=0.9796 fold_std=0.0480 pr=0.9069 (37s)
LGBM | raw + rs smote x8 [3 seeds]               dev=0.9543 gap=0.9146 roll=0.9808 fold_std=0.0447 pr=0.9078 (41s)
LGBM | raw + rs smote x12 [3 seeds]              dev=0.9514 gap=0.9109 roll=0.9785 fold_std=0.0492 pr=0.9058 (60s)
LGBM | raw + rs smote x4 cross-step [3 seeds]    dev=0.9446 gap=0.8962 roll=0.9768 fold_std=0.0568 pr=0.9032 (70s)
LGBM | raw + rs smote x8 cross-step [3 seeds]    dev=0.9478 gap=0.9015 roll=0.9787 fold_std=0.0535 pr=0.9040 (61s)
reference LGBM | raw [3 seeds]: dev=0.9448 gap=0.8979
                                        model  roc_dev  roc_gap  fold_std   d_dev   d_gap
           LGBM | raw + rs smot

{'model': 'LGBM | stack4 rs final smote x4 [3 seeds]',
 'roc_dev': np.float64(0.9490108491514085),
 'roc_roll': np.float64(0.975422939409388),
 'roc_gap': np.float64(0.9093927137644398),
 'fold_std': np.float64(0.04786296418696597),
 'seed_std': np.float64(0.006230527317030015),
 'pr_dev': np.float64(0.9055213692201006),
 'prec': np.float64(0.9341517214596837),
 'rec': np.float64(0.8075179876173794),
 'f1': np.float64(0.8607515502615593),
 'f1_best': np.float64(0.8783069739914376),
 'secs': 78}

### Findings
References over 3 seeds: plain LightGBM dev 0.9448 / gap 0.8979; stack2 dev 0.9478 / gap 0.9059. Every method works inside each training window, one time step at a time.

| Method | Plain: dev / gap | In stack2 (final stage): dev / gap |
|---|---|---|
| undersample 1:1 | 0.9440 / 0.8951 | not run |
| undersample 3:1 | 0.9425 / 0.8934 | 0.9478 / 0.9057 |
| undersample 10:1 | 0.9463 / 0.8985 | not run |
| undersample 3:1: uniform rate / reweighted / 8 bags | 0.9452 / 0.8967; 0.9464 / 0.8982; 0.9450 / 0.8955 | not run |
| undersample 3:1, recency 0.9 | 0.9396 / 0.8855 | 0.9442 / 0.8972 |
| oversample (duplicate) x2 | 0.9423 / 0.8902 | 0.9499 / 0.9081 |
| oversample (duplicate) x4 | 0.9411 / 0.8889 | not run |
| oversample x4, recency 0.9 | 0.9381 / 0.8790 | not run |
| per-step SMOTE x2 | 0.9496 / 0.9055 | 0.9504 / 0.9135 |
| **per-step SMOTE x4** | **0.9539 / 0.9183** | 0.9490 / 0.9094 |
| per-step SMOTE x4, recency 0.9 | 0.9531 / 0.9150 | not run |
| per-step SMOTE x6 / x8 / x12 | 0.9530 / 0.9132; 0.9543 / 0.9146; 0.9514 / 0.9109 | not run |
| SMOTE across steps, x4 / x8 | 0.9446 / 0.8962; 0.9478 / 0.9015 | not run |
| undersample 3:1 + SMOTE x2 | 0.9459 / 0.8970 | 0.9491 / 0.9077 |
| resampling in stage 1 (undersample 3:1 / SMOTE x2) | not applicable | 0.9468 / 0.9022; 0.9470 / 0.9009 |

* **Only per-step SMOTE helps consistently.** Every multiplier from x2 to x12 improves the gap folds (+0.008 to +0.020), and x4 to x8 are within seed noise of each other, so the multiplier was not tuned further.
* **It is the interpolation, not the class balance.** Duplicating illicit rows to the same count hurts (gap -0.008 to -0.009), while SMOTE helps.
* **Interpolation must stay inside a time step.** SMOTE across steps loses almost all of the gain (x4 gap -0.0017 against +0.0204 per step).
* **Undersampling does nothing useful.** Every variant is within +-0.005 of the reference on the gap folds. Recency weighting made every method worse except SMOTE, where it was slightly lower (0.9150 against 0.9183), so it is not used.
* **SMOTE and stacking do not add up.** In stack2 the gain shrinks (x2 +0.0076 on gap, x4 +0.0035), SMOTE x4 loses 6 of 8 folds, and resampling stage 1 hurts. The final SMOTE model is plain LightGBM.
* **8-fold check.** Undersampling 3:1 loses 2 of 8 folds in both settings and is rejected. Plain SMOTE x4 wins 6 of 8 (gap +0.0204, dev +0.0090, mean +0.0063). It misses the 7-of-8 rule by one fold, with the two losses at -0.0000 and -0.0017. Gap fold 1 supplies most of the gain (+0.032).
* **Leaderboard tie-break.** Plain LightGBM with per-step SMOTE x4 (3 seeds) scored **0.95591**, against 0.95157 for stack2 and 0.94894 for plain LightGBM (Public leaderboard). A rank average of SMOTE and stack2 scored 0.95469, below SMOTE alone.

## 11. Frozen holdout, threshold tuning, error analysis

Only the finalists are scored on the holdout (train steps 1-25, validate 31-35).

In [49]:
SIM_VAL_MASK = 0.25     # validation / holdout masking (set to 0.0 only for the final fit in section 12)

In [ ]:
# Register finalists: name -> fp
FINALISTS = {
    BASELINE: gbdt_fp(MAIN_FACTORY, X_BASE, decay=DECAY),
    "XGB | raw": gbdt_fp(make_xgb, X_BASE, decay=DECAY),
    "LGBM | stack2: train mask .25, 25% val masked":
        stacked_fp2(tree_stage1(make_hgb, X_BASE), X_BASE, MAIN_FACTORY, decay=DECAY, p_mask=0.25),
    "LGBM | raw + per-step SMOTE x4":
        rs_gbdt_fp(MAIN_FACTORY, X_BASE, SAMPLERS["smote x4"]),
}

for n, fp in FINALISTS.items():
    holdout_eval(n, fp, seeds=(0, 1, 2))

# ensemble of finalists on the holdout (rank average)
ens_names = [n for n in FINALISTS]
y_h = Y[fold_pos(HOLDOUT)[1]].astype(int)
ens_h = rank_avg([PRED[n]["holdout"] for n in ens_names])
print(f"[HOLDOUT] ensemble of finalists: roc={roc_auc_score(y_h, ens_h):.4f} pr={average_precision_score(y_h, ens_h):.4f}")
pd.DataFrame(HOLD)

[HOLDOUT] LGBM | raw                                   roc=0.9969 pr=0.9872 f1_best=0.9566
[HOLDOUT] XGB | raw                                    roc=0.9951 pr=0.9778 f1_best=0.9331
[HOLDOUT] LGBM | stack2: train mask .25, 25% val masked roc=0.9970 pr=0.9891 f1_best=0.9685
[HOLDOUT] ensemble of finalists: roc=0.9971 pr=0.9870


,model,roc_auc,pr_auc,prec,rec,f1,f1_best
0,LGBM | stack2 + shallow final,0.996972,0.988515,0.953757,0.956522,0.955137,0.963397
1,LGBM | raw,0.996923,0.987210,0.981279,0.911594,0.945154,0.956586
2,XGB | raw,0.995141,0.977752,0.976654,0.727536,0.833887,0.933142
3,"LGBM | stack2: train mask .25, 25% val masked",0.996998,0.989104,0.980510,0.947826,0.963891,0.968452


### Findings
LightGBM 0.9969 (PR-AUC 0.9872), XGBoost 0.9951 (0.9778), stack2 0.9970 (0.9891), rank average of the three 0.9971. Stack2 is better on PR-AUC and best-threshold F1 (0.9685 vs 0.9566), which is the illicit-class precision and recall side of the task, and equal on ROC-AUC. The rank average of the three has a lower PR-AUC (0.9870) than stack2 alone (0.9891). The table also lists the stack2 + shallow final model from section 10.2 (ROC 0.9970, PR-AUC 0.9885, best-threshold F1 0.9634), which is no better than stack2.

### Threshold tuning
Pick `t` on **validation** predictions to maximise illicit F1, then fix that `t` and apply it unchanged to data it was not tuned on. Here `t` is chosen on the pooled dev-fold predictions and applied to the holdout.

The threshold analysis was done on stack2 only. The final SMOTE model (sub5) is trained on rebalanced data, so its probabilities are shifted upward and the stack2 threshold does not apply to it.

In [51]:
name = "LGBM | stack2: train mask .25, 25% val masked"

# Step 1: pool the dev-fold predictions of that model
dev_y, dev_p = [], []
for s in ("roll", "gap"):
    for k, f in enumerate(DEV_FOLDS[s]):
        _, va = fold_pos(f); dev_y.append(Y[va].astype(int)); dev_p.append(PRED[name][(s, k)])
dev_y, dev_p = np.concatenate(dev_y), np.concatenate(dev_p)

# Step 2: find the cut-off that gives the best F1 on the dev folds
ts = np.linspace(0.02, 0.98, 97)
f1s = [illicit_prf(dev_y, dev_p, t)[2] for t in ts]
t_star = ts[int(np.argmax(f1s))]

# Step 3: test it on the holdout
ph = PRED[name]["holdout"]
print(f"model: {name}")
print(f"best threshold on dev folds: {t_star:.2f}  (dev F1 {max(f1s):.3f})")
for t, lab in ((0.5, "default 0.5"), (t_star, f"tuned {t_star:.2f}")):
    p, r, f = illicit_prf(y_h, ph, t)
    print(f"holdout @ {lab:14s} precision={p:.3f} recall={r:.3f} F1={f:.3f}")

model: LGBM | stack2: train mask .25, 25% val masked
best threshold on dev folds: 0.21  (dev F1 0.857)
holdout @ default 0.5    precision=0.981 recall=0.948 F1=0.964
holdout @ tuned 0.21     precision=0.941 recall=0.967 F1=0.954


In [52]:
def per_step_report(name):
    _, va = fold_pos(HOLDOUT)
    df = pd.DataFrame({"step": T[va], "y": Y[va].astype(int), "s": PRED[name]["holdout"]})
    rows = []
    for t, g in df.groupby("step"):
        two = g["y"].nunique() == 2
        rows.append(dict(step=t, n=len(g), illicit=int(g["y"].sum()),
                         roc_auc=roc_auc_score(g["y"], g["s"]) if two else np.nan,
                         pr_auc=average_precision_score(g["y"], g["s"]) if two else np.nan))
    return pd.DataFrame(rows)

per_step_report("LGBM | stack2: train mask .25, 25% val masked")

,step,n,illicit,roc_auc,pr_auc
0,31,710,106,0.991644,0.959994
1,32,1323,342,0.999198,0.997532
2,33,441,23,0.992615,0.928223
3,34,515,37,0.999378,0.992573
4,35,1341,182,0.997065,0.987997


In [53]:
# Permutation importance of the baseline model on the holdout (diagnostic).
tr_h, va_h = fold_pos(HOLDOUT)
m_imp = fit_model(make_hgb, X_BASE[tr_h], Y[tr_h].astype(int), None, 0, False)
pi = permutation_importance(m_imp, X_BASE[va_h], Y[va_h].astype(int), scoring="roc_auc", n_repeats=3, random_state=0, n_jobs=-1)
names_base = feat if X_BASE.shape[1] == len(feat) else [f"f{i}" for i in range(X_BASE.shape[1])]
pd.Series(pi.importances_mean, index=names_base).sort_values(ascending=False).head(15)

feat_59     0.040248
feat_53     0.005917
feat_46     0.005802
feat_5      0.002395
feat_163    0.002271
feat_125    0.001964
feat_76     0.001952
feat_90     0.001731
feat_80     0.001012
feat_161    0.000582
feat_52     0.000530
feat_47     0.000444
feat_3      0.000379
feat_61     0.000251
feat_132    0.000241
dtype: float64

### Findings
* **Threshold tuning does not transfer.** The F1-optimal threshold on the pooled dev folds is 0.21 (dev F1 0.857). Applied to the holdout it trades precision for recall (precision 0.981 to 0.941, recall 0.948 to 0.967) and lowers F1 from 0.964 (default 0.5) to 0.954. The default 0.5 is kept.
* **Per-step scores on the holdout** (stack2) are high in every step: ROC-AUC 0.992 to 0.999. The weakest are step 31 (0.9916) and step 33 (0.9926). Step 33 has the lowest PR-AUC (0.928) and only 23 illicit rows, so its score is the least reliable.
* **Permutation importance** (HistGradientBoosting model, holdout ROC-AUC drop) is dominated by one feature: `feat_59` (0.040), far ahead of `feat_53` (0.0059) and `feat_46` (0.0058). A few strong features carry most of the signal, as in the column-sampling results of section 10.1.

## 12. Final fit and submission files

All model selection above was done locally. The final models are fitted on **all** labelled rows (steps 1-35) and predict the test rows through the same `fp(tr, va, seed)` interface used in validation, with `SIM_VAL_MASK = 0.0` and `QUICK = True` (3 seeds). The cell writes two files, each with scores rescaled to ranks in (0, 1] and the exact row order of `sample_submission.csv`:

| File | Model | Public ROC-AUC |
|---|---|---|
| `sub5_smote_x4.csv` | LightGBM on the raw features with per-time-step SMOTE x4 | 0.95591 |
| `sub6_blend_stack2_smote.csv` | rank average of stack2 and sub5 | 0.95469 |

The earlier files (sub1 baseline 0.94894, sub2 stack2 0.95157, sub3 blend 0.95094) are still in the cell but commented out, since they scored lower. The final entries are **sub5** and **sub6**.

In [ ]:
SIM_VAL_MASK = 0.0
FINAL_SEEDS = (0, 1, 2) if QUICK else (0, 1, 2, 3, 4)
assert QUICK, "the leaderboard scores of sub5/sub6 (0.95591 / 0.95469) were produced with QUICK=True"
TEST_PRED = {}

def predict_test(name, fp):
    if name not in TEST_PRED:
        TEST_PRED[name] = np.mean([np.asarray(fp(LAB, TEST_POS, sd), dtype=np.float64) for sd in FINAL_SEEDS], axis=0)
        print("fitted:", name)
    return TEST_PRED[name]

def write_submission(scores, path):
    sub = sample_sub.copy()
    sub["target"] = scores
    assert len(sub) == len(sample_sub) and (sub["index"] == sample_sub["index"]).all()
    assert not sub["target"].isna().any()
    sub.to_csv(path, index=False)
    print(f"wrote {path}: rows={len(sub)} range=({sub['target'].min():.4f}, {sub['target'].max():.4f})")
    return sub

STACK2_NAME = "LGBM | stack2: train mask .25, 25% val masked"
SMOTE_NAME  = "LGBM | raw + per-step SMOTE x4"

# earlier entries (scored lower on the leaderboard, no longer written)
# BASE_NAME = f"{MAIN} | raw (plain baseline)"
# s1 = predict_test(BASE_NAME, gbdt_fp(MAIN_FACTORY, X_raw))
# sub1 = write_submission(rankdata(s1) / len(s1), f"{OUT_DIR}/sub1_baseline.csv")        # 0.94894
# sub2 = write_submission(rankdata(s2) / len(s2), f"{OUT_DIR}/sub2_stack2.csv")          # 0.95157
# s3 = rank_avg([s1, s2])
# sub3 = write_submission(s3, f"{OUT_DIR}/sub3_blend.csv")                                # 0.95094

# final entries
s2 = predict_test(STACK2_NAME, FINALISTS[STACK2_NAME])         # only used inside sub6
s5 = predict_test(SMOTE_NAME, rs_gbdt_fp(MAIN_FACTORY, X_raw, SAMPLERS["smote x4"]))

sub5 = write_submission(rankdata(s5) / len(s5), f"{OUT_DIR}/sub5_smote_x4.csv")          # public 0.95591
s6 = rank_avg([s2, s5])
sub6 = write_submission(s6, f"{OUT_DIR}/sub6_blend_stack2_smote.csv")                    # public 0.95469

from scipy.stats import spearmanr
print("spearman stack2 vs smote:", round(spearmanr(s2, s5)[0], 4))

leaderboard().to_csv(f"{OUT_DIR}/experiment_log.csv", index=False)
pd.DataFrame(HOLD).to_csv(f"{OUT_DIR}/holdout_log.csv", index=False)

fitted: LGBM | raw (plain baseline)
wrote c:\Users\erina\Downloads\bt4012_kaggle competition/sub1_baseline.csv: rows=15329 range=(0.0001, 1.0000)
fitted: LGBM | stack2: train mask .25, 25% val masked
wrote c:\Users\erina\Downloads\bt4012_kaggle competition/sub2_stack2.csv: rows=15329 range=(0.0001, 1.0000)
wrote c:\Users\erina\Downloads\bt4012_kaggle competition/sub3_blend.csv: rows=15329 range=(0.0001, 0.9999)
spearman sub1 vs sub2: 0.9475


## 13. Decision log

Dev and gap are mean ROC-AUC over the 5 dev folds and the 2 gap folds. Single-seed rows are marked (1s); the rest use 3 seeds.

| Step | Change | Dev | Gap | Kept? | Public score |
|---|---|---|---|---|---|
| 1 | LightGBM on raw features (1s / 3 seeds) | 0.9490 / 0.9448 | 0.9048 / 0.8979 | **no (superseded by sub5)** | 0.94894 |
| 2 | Class weighting (1s) | 0.9379 | 0.8752 | no | |
| 3 | Rank normalisation (1s) | 0.9361 | 0.8995 | no | |
| 4 | Recency weights 0.9 / 0.95 (1s) | 0.9496 / 0.9508 | 0.9104 / 0.9103 | no (2 of 5 folds) | |
| 5 | Drop top 6 / 20 drifting features (1s) | 0.9462 / 0.9488 | 0.9021 / 0.9036 | no | |
| 6 | Degree / neighbour-structure features (1s) | 0.9484 / 0.9478 | 0.9065 / 0.9087 | no (noise) | |
| 7 | PageRank / k-core features (1s) | 0.9427 / 0.9408 | 0.8891 / 0.8867 | no (hurt) | |
| 8 | **Neighbour-score stacking, stack2** | 0.9478 | 0.9059 | **used in sub6 blend** | 0.95157 |
| 9 | Self-training (1s) | 0.9246 | 0.8649 | no | |
| 10 | Tree-model ensembles (LightGBM + XGBoost + stack2) | | 0.9016 | no (sub3 blend) | 0.95094 |
| 11 | Best tuned LightGBM setting (shallow, column sampling 1.0) | 0.9461 | 0.9019 | no (noise) | |
| 12 | Local features only | 0.9422 | 0.8915 | no | |
| 13 | Stack2 + shallow final | 0.9511 | 0.9128 | no | 0.94725 |
| 14 | Richer neighbour summaries | 0.9472 | 0.9046 | no | |
| 15 | Component features | 0.9519 | 0.9140 | no (drifts) | |
| 16 | Score propagation | 0.9479 | 0.9093 | no | |
| 17 | All three together | 0.9462 | 0.9040 | no (3 of 8 folds) | |
| 18 | Diverse stage 1 | 0.9498 | 0.9094 | no | |
| 19 | Second stacking round | 0.9558 | 0.9226 | no (4 of 8 folds) | |
| 20 | Unknown rows as weak negatives (weight 0.03) | 0.9248 | 0.8617 | no | |
| 21 | Sliding window (8 steps) | 0.9458 | 0.9003 | no | |
| 22 | Balanced bagging (3:1) | 0.9460 | 0.8966 | no | |
| 23 | Edge-dropout training (with row masking) | 0.9471 | 0.9050 | no | |
| 24 | GCN / GraphSAGE (1s) | 0.9106 / 0.9362 | 0.8638 / 0.8949 | no | |
| 25 | Stack2 + SAGE blend (weight 0.2) | 0.9482 | 0.9062 | no | |
| 26 | More trees, 500 / 800 instead of 300 (plain LightGBM) | 0.9450 / 0.9423 | 0.8960 / 0.8909 | no (flat, then falls) | |
| 27 | More trees, 500 / 800 instead of 300 (stack2) | 0.9489 / 0.9472 | 0.9061 / 0.9028 | no | |
| 28 | Tuned decision threshold, 0.21 instead of 0.5 (holdout F1 0.954 vs 0.964) | | | no | |
| 29 | 2-hop shared-neighbour aggregates | 0.9473 | 0.9057 | no (5 of 8 folds) | |
| 30 | 2-hop directed aggregates | | | no (4 of 8 folds) | |
| 31 | Neighbour means of the top-X raw features | | | no (X=5 fails 4 of 8 folds) | |
| 32 | Undersampling (1:1, 3:1, 10:1) and duplication | 0.9425 / 0.9423 | 0.8934 / 0.8902 | no | |
| 33 | **Per-step SMOTE x4, plain LightGBM** | 0.9539 | 0.9183 | **yes (sub5)**, 6 of 8 folds | 0.95591 |
| 34 | SMOTE x4 in stack2 final stage | 0.9490 | 0.9094 | no (2 of 8 folds) | |
| 35 | Rank average of stack2 and sub5 | | | **yes (sub6)** | 0.95469 |

Local validation noise (fold std about 0.05, seed std about 0.005) is larger than most of the gains tested, so the local 7-of-8 rule is a guide only. SMOTE x4 missed it by one fold but gave the best public score (0.95591), and was kept.

## Appendix: experimental file `sub4`
Written for the record only (stack2 with the shallow final model, 5 seeds). It scored **0.94725** and is not used. The cell below is commented out; it needs fp_new from section 10.2.

In [ ]:
# sub4 (experimental, rejected, public 0.94725) is no longer written. Uncomment to reproduce; needs fp_new from section 10.2.

# SIM_VAL_MASK = 0.0
# FINAL_SEEDS = (0, 1, 2, 3, 4)

# s4 = np.mean([np.asarray(fp_new(LAB, TEST_POS, sd), dtype=np.float64) for sd in FINAL_SEEDS], axis=0)
# sub4 = sample_sub.copy(); sub4["target"] = rankdata(s4) / len(s4)
# assert len(sub4) == len(sample_sub) and (sub4["index"] == sample_sub["index"]).all()
# assert not sub4["target"].isna().any()
# sub4.to_csv(f"{OUT_DIR}/sub4_stack2_shallow.csv", index=False)
# print("wrote sub4, rows:", len(sub4))

wrote sub4, rows: 15329
